# Saddle-Mediated Transport — Sigmoid Projection Along the Transit Axis

**Hypothesis.** Training dynamics pass through a saddle. The saddle axis is rotated from the principal components of the parameter trajectory. One end of the axis is the *wiggle-point* (the deflection right after first descent, which coincides with a peak in MLP dimensionality). The other end is the final checkpoint.

If that picture is right, projecting the full-parameter trajectory onto the unit vector `(terminal − wiggle)` should yield a sigmoid over epochs. Its derivative should peak at the saddle crossing.

**Reference variant.** `p109/s485/ds598` — reference healthy model with dense checkpointing (every 100 epochs through epoch 10000). `first_descent_end=1200`, `second_descent_onset=3584`, `grokking=5243`.

**Parameter space.** MLP only — `W_in` + `W_out` flattened and concatenated (the `COMPONENT_GROUPS['mlp']` bundle). Attention and embedding trajectories don't show the deflection, so scoping to MLP keeps the signal uncontaminated.

In [ ]:
import json
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from miscope import load_family
from miscope.analysis.artifact_loader import ArtifactLoader
from miscope.analysis.library.weights import COMPONENT_GROUPS
from miscope.visualization.renderers.dimensionality_dynamics import (
    _compute_rolling_trajectory_metrics,
)

FAMILY_NAME = 'modulo_addition_1layer'
family = load_family(FAMILY_NAME)

PRIME, SEED, DSEED = 109, 485, 598
variant = family.get_variant(prime=PRIME, seed=SEED, data_seed=DSEED)
label = f'p{PRIME}/s{SEED}/ds{DSEED}'

with open(variant.variant_dir / 'variant_summary.json') as f:
    summary = json.load(f)
TIMING = {
    'fd_end':  summary['first_descent_window']['end_epoch'],
    'sd_onset': summary.get('second_descent_onset_epoch'),
    'grok':     summary.get('test_loss_threshold_first_epoch'),
}
print(label, TIMING)

## 1. Load MLP parameter trajectory — full parameter space

`parameter_snapshot` stores all 9 weight matrices per epoch. For each checkpoint we flatten and concatenate `W_in` and `W_out` into a single vector, yielding `X` of shape `(n_epochs, d_mlp)`. No PCA reduction — the transit vector must live in the full space because the saddle axis is rotated from the principal directions.

In [ ]:
def load_mlp_param_trajectory(variant):
    """Load W_in ⊕ W_out flattened per epoch — full MLP parameter space."""
    loader = ArtifactLoader(str(variant.variant_dir / 'artifacts'))
    epochs = sorted(loader.get_epochs('parameter_snapshot'))
    mlp_keys = COMPONENT_GROUPS['mlp']  # ['W_in', 'W_out']
    rows = []
    for e in epochs:
        snap = loader.load_epoch('parameter_snapshot', e)
        rows.append(np.concatenate([snap[k].flatten() for k in mlp_keys]))
    X = np.stack(rows).astype(np.float64)
    return np.array(epochs), X


epochs, X = load_mlp_param_trajectory(variant)
print(f'n_epochs = {len(epochs)}, d_mlp = {X.shape[1]}')
print(f'epoch range: {epochs[0]} → {epochs[-1]}')

## 2. Wiggle-point — peak of rolling PR₃ on the MLP trajectory

Apply `_compute_rolling_trajectory_metrics` (same function the dimensionality view uses) to the precomputed `mlp__projections`. Search for the PR₃ peak in the window `[fd_end, grok]`. Also compute the first PC3 extremum as a cross-check (your option 2).

In [ ]:
loader = ArtifactLoader(str(variant.variant_dir / 'artifacts'))
pt = loader.load_cross_epoch('parameter_trajectory')
pt_epochs = pt['epochs']
mlp_proj = pt['mlp__projections']  # (n_epochs, 10)

pr3_mlp, ft_mlp = _compute_rolling_trajectory_metrics(mlp_proj, window=10)

# Peak in [fd_end, grok]
search_lo = TIMING['fd_end']
search_hi = TIMING['grok']
in_window = (pt_epochs >= search_lo) & (pt_epochs <= search_hi)
pr3_masked = np.where(in_window, pr3_mlp, -np.inf)
wiggle_idx_pr3 = int(np.argmax(pr3_masked))
wiggle_epoch_pr3 = int(pt_epochs[wiggle_idx_pr3])

# Cross-check: first PC3 extremum in same window (earliest |PC3| peak post-fd_end)
pc3 = mlp_proj[:, 2]
pc3_search = np.where(in_window, np.abs(pc3), -np.inf)
wiggle_idx_pc3 = int(np.argmax(pc3_search))
wiggle_epoch_pc3 = int(pt_epochs[wiggle_idx_pc3])

print(f'Rolling-PR₃ peak in [{search_lo}, {search_hi}] → epoch {wiggle_epoch_pr3}  (PR₃={pr3_mlp[wiggle_idx_pr3]:.3f})')
print(f'|PC3|    peak in [{search_lo}, {search_hi}] → epoch {wiggle_epoch_pc3}  (PC3={pc3[wiggle_idx_pc3]:+.3f})')

In [ ]:
# Visualize: rolling PR₃ and |PC3| with candidate wiggle-points
fig = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.08,
                    subplot_titles=['MLP trajectory — rolling PR₃',
                                    'MLP trajectory — PC3 coordinate'])
fig.add_trace(go.Scatter(x=pt_epochs, y=pr3_mlp, mode='lines', name='PR₃',
                         line=dict(color='#d62728', width=2)), row=1, col=1)
fig.add_trace(go.Scatter(x=pt_epochs, y=pc3, mode='lines', name='PC3',
                         line=dict(color='#1f77b4', width=2)), row=2, col=1)

for row in (1, 2):
    fig.add_vline(x=TIMING['fd_end'], line=dict(color='orange', dash='dot', width=1), row=row, col=1)
    fig.add_vline(x=TIMING['grok'],   line=dict(color='black',  dash='dash', width=1), row=row, col=1)
    fig.add_vline(x=wiggle_epoch_pr3, line=dict(color='red',    dash='solid', width=1.5), row=row, col=1)
    fig.add_vline(x=wiggle_epoch_pc3, line=dict(color='purple', dash='solid', width=1.5), row=row, col=1)

fig.update_yaxes(title_text='PR₃', range=[0.9, 3.1], row=1, col=1)
fig.update_yaxes(title_text='PC3', row=2, col=1)
fig.update_xaxes(title_text='epoch', row=2, col=1)
fig.update_layout(
    title=f'{label} — wiggle-point candidates<br><sup>orange=fd_end · black=grok · red=PR₃ peak · purple=|PC3| peak</sup>',
    template='plotly_white', height=600, width=1000,
)
fig.show()

## 3. Transit vector and scalar projection

For a chosen wiggle epoch `e_w` and terminal epoch `e_T`:

```
v = (X[e_T] − X[e_w]) / ‖X[e_T] − X[e_w]‖
s(t) = (X[t] − X[e_w]) · v     ≡     scalar distance along v from wiggle
```

Normalizing by `‖X[e_T] − X[e_w]‖` so `s=0` at wiggle and `s=1` at terminal. If the saddle picture is right, `s(t)` is sigmoidal with inflection at the crossing.

In [ ]:
def compute_transit_projection(X, epochs, wiggle_epoch, terminal_epoch):
    """Project full trajectory onto unit transit vector; normalize so wiggle→0, terminal→1."""
    i_w = int(np.where(epochs == wiggle_epoch)[0][0])
    i_T = int(np.where(epochs == terminal_epoch)[0][0])
    delta = X[i_T] - X[i_w]
    L = np.linalg.norm(delta)
    v_hat = delta / L
    # s normalized so s=0 at wiggle, s=1 at terminal
    s = (X - X[i_w]) @ v_hat / L
    return s, L, i_w, i_T


def compute_velocity(s, epochs):
    """ds/dt using central differences scaled by per-epoch gap."""
    s = np.asarray(s, dtype=np.float64)
    ep = np.asarray(epochs, dtype=np.float64)
    ds_dt = np.zeros_like(s)
    ds_dt[1:-1] = (s[2:] - s[:-2]) / (ep[2:] - ep[:-2])
    ds_dt[0]    = (s[1] - s[0]) / (ep[1] - ep[0])
    ds_dt[-1]   = (s[-1] - s[-2]) / (ep[-1] - ep[-2])
    return ds_dt


WIGGLE_EPOCH = wiggle_epoch_pr3           # primary candidate — rolling PR₃ peak
TERMINAL_EPOCH = int(epochs[-1])

s, L, i_w, i_T = compute_transit_projection(X, epochs, WIGGLE_EPOCH, TERMINAL_EPOCH)
v = compute_velocity(s, epochs)

saddle_idx = int(np.argmax(v))
saddle_epoch = int(epochs[saddle_idx])
print(f'wiggle epoch:     {WIGGLE_EPOCH}')
print(f'terminal epoch:   {TERMINAL_EPOCH}')
print(f'transit length:   {L:.3f}  (‖W_T − W_w‖ in MLP space)')
print(f'velocity-peak ep: {saddle_epoch}  (ds/dt max = {v[saddle_idx]:.2e})')
print(f's at wiggle = {s[i_w]:.3f}, s at terminal = {s[i_T]:.3f}  (should be 0.0 and 1.0)')

In [ ]:
# Sigmoid + velocity plot
fig = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.08,
                    subplot_titles=['s(t)  —  scalar projection along transit axis  (0=wiggle, 1=terminal)',
                                    'ds/dt  —  projection velocity'])

fig.add_trace(go.Scatter(x=epochs, y=s, mode='lines+markers', name='s(t)',
                         line=dict(color='#1f77b4', width=2),
                         marker=dict(size=4)), row=1, col=1)
fig.add_hline(y=0.0, line=dict(color='gray', dash='dot', width=1), row=1, col=1)
fig.add_hline(y=1.0, line=dict(color='gray', dash='dot', width=1), row=1, col=1)
fig.add_hline(y=0.5, line=dict(color='rgba(0,0,0,0.15)', dash='dot', width=1), row=1, col=1)

fig.add_trace(go.Scatter(x=epochs, y=v, mode='lines+markers', name='ds/dt',
                         line=dict(color='#d62728', width=2),
                         marker=dict(size=4)), row=2, col=1)

for row in (1, 2):
    fig.add_vline(x=TIMING['fd_end'],    line=dict(color='orange', dash='dot', width=1), row=row, col=1)
    fig.add_vline(x=TIMING['sd_onset'],  line=dict(color='teal',   dash='dot', width=1), row=row, col=1)
    fig.add_vline(x=TIMING['grok'],      line=dict(color='black',  dash='dash', width=1), row=row, col=1)
    fig.add_vline(x=WIGGLE_EPOCH,        line=dict(color='red',    dash='solid', width=1.5), row=row, col=1)
    fig.add_vline(x=saddle_epoch,        line=dict(color='green',  dash='solid', width=1.5), row=row, col=1)

fig.update_yaxes(title_text='s (normalized)', row=1, col=1)
fig.update_yaxes(title_text='ds/dt',         row=2, col=1)
fig.update_xaxes(title_text='epoch', row=2, col=1)
fig.update_layout(
    title=(f'{label} — transit projection  (wiggle ep={WIGGLE_EPOCH}, terminal ep={TERMINAL_EPOCH})<br>'
           '<sup>orange=fd_end · teal=sd_onset · black=grok · red=wiggle · green=ds/dt peak</sup>'),
    template='plotly_white', height=700, width=1000, showlegend=False,
)
fig.show()

## 4. Wiggle-point sensitivity — PR₃ peak vs |PC3| peak

Overlay `s(t)` for both wiggle-point choices against the same terminal. If both give the same sigmoid shape (up to an affine rescaling of the axis), the two operationalizations agree on the saddle picture.

In [ ]:
WIGGLE_CANDIDATES = [
    ('PR₃ peak',   wiggle_epoch_pr3, '#1f77b4'),
    ('|PC3| peak', wiggle_epoch_pc3, '#9467bd'),
]

fig = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.08,
                    subplot_titles=['s(t) per wiggle-point choice',
                                    'ds/dt per wiggle-point choice'])

for name, we, color in WIGGLE_CANDIDATES:
    s_c, L_c, _, _ = compute_transit_projection(X, epochs, we, TERMINAL_EPOCH)
    v_c = compute_velocity(s_c, epochs)
    fig.add_trace(go.Scatter(x=epochs, y=s_c, mode='lines', name=f'{name} (ep {we})',
                             line=dict(color=color, width=2)), row=1, col=1)
    fig.add_trace(go.Scatter(x=epochs, y=v_c, mode='lines', name=f'{name} (ep {we})',
                             line=dict(color=color, width=2), showlegend=False), row=2, col=1)

for row in (1, 2):
    fig.add_vline(x=TIMING['fd_end'], line=dict(color='orange', dash='dot', width=1), row=row, col=1)
    fig.add_vline(x=TIMING['grok'],   line=dict(color='black',  dash='dash', width=1), row=row, col=1)

fig.update_yaxes(title_text='s', row=1, col=1)
fig.update_yaxes(title_text='ds/dt', row=2, col=1)
fig.update_xaxes(title_text='epoch', row=2, col=1)
fig.update_layout(
    title=f'{label} — wiggle-point sensitivity  (terminal ep={TERMINAL_EPOCH})',
    template='plotly_white', height=700, width=1000,
)
fig.show()

## 5. Terminal sensitivity — does the sigmoid shift under truncation?

If MLP dimensionality is cyclic (your hypothesis), the choice of terminal may distort the transit axis. Overlay `s(t)` for terminal choices at 10k, 15k, and final (25k) — using the PR₃-peak wiggle. A stable sigmoid under all three says the saddle axis is robust; drift says we need to re-examine the terminal.

In [ ]:
def closest_epoch(epochs_arr, target):
    return int(epochs_arr[int(np.argmin(np.abs(epochs_arr - target)))])

TERMINAL_CANDIDATES = [
    ('terminal 10k',  closest_epoch(epochs, 10000)),
    ('terminal 15k',  closest_epoch(epochs, 15000)),
    ('terminal final', int(epochs[-1])),
]

fig = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.08,
                    subplot_titles=['s(t) per terminal choice  (wiggle = PR₃ peak)',
                                    'ds/dt per terminal choice'])
colors = ['#2ca02c', '#ff7f0e', '#d62728']

for (name, te), color in zip(TERMINAL_CANDIDATES, colors):
    s_c, L_c, _, _ = compute_transit_projection(X, epochs, wiggle_epoch_pr3, te)
    v_c = compute_velocity(s_c, epochs)
    fig.add_trace(go.Scatter(x=epochs, y=s_c, mode='lines',
                             name=f'{name} (ep {te}, L={L_c:.2f})',
                             line=dict(color=color, width=2)), row=1, col=1)
    fig.add_trace(go.Scatter(x=epochs, y=v_c, mode='lines', name=name,
                             line=dict(color=color, width=2), showlegend=False), row=2, col=1)

for row in (1, 2):
    fig.add_vline(x=TIMING['fd_end'], line=dict(color='orange', dash='dot', width=1), row=row, col=1)
    fig.add_vline(x=TIMING['grok'],   line=dict(color='black',  dash='dash', width=1), row=row, col=1)
    fig.add_vline(x=wiggle_epoch_pr3, line=dict(color='red',    dash='solid', width=1), row=row, col=1)

fig.update_yaxes(title_text='s', row=1, col=1)
fig.update_yaxes(title_text='ds/dt', row=2, col=1)
fig.update_xaxes(title_text='epoch', row=2, col=1)
fig.update_layout(
    title=f'{label} — terminal sensitivity  (wiggle ep={wiggle_epoch_pr3})',
    template='plotly_white', height=700, width=1000,
)
fig.show()

## 6. Checkpointing resilience — subsample test

Take p109's dense 100-epoch trajectory and subsample it to mimic what sparse-checkpointed variants look like. Two scenarios tested:

**(a) Uniformly sparse** — every-500 spacing throughout. Matches the typical 'plateau-sparse' pattern on existing variants.

**(b) Hybrid** — dense (every 100) through a cutoff, sparse (every 500) after. Matches what's realistic for new training runs: dense checkpointing through the grokking region, then sparse post-grok.

For each pattern we refit PCA on the subsampled snapshots (simulating what the `parameter_trajectory` analyzer would have stored) and re-run the full pipeline. Metrics tracked vs dense baseline: wiggle epoch shift, saddle epoch shift, transit length Δ.

In [ ]:
from sklearn.decomposition import PCA


def identify_wiggle(X, epochs_arr, fd_end, grok):
    """Refit PCA, compute rolling PR₃, find peak in [fd_end, grok]."""
    pca = PCA(n_components=min(10, len(X) - 1)).fit(X)
    proj = pca.transform(X)
    pr3, _ = _compute_rolling_trajectory_metrics(proj, window=min(10, len(X)))
    in_win = (epochs_arr >= fd_end) & (epochs_arr <= grok)
    if in_win.sum() == 0:
        return None
    return int(epochs_arr[int(np.argmax(np.where(in_win, pr3, -np.inf)))])


def full_pipeline(X, eps, fd_end, grok):
    we = identify_wiggle(X, eps, fd_end, grok)
    if we is None:
        return None
    s, L, _, _ = compute_transit_projection(X, eps, we, int(eps[-1]))
    v = compute_velocity(s, eps)
    return dict(
        wiggle=we, L=L,
        saddle=int(eps[int(np.argmax(v))]),
        max_dsdt=float(v.max()),
        s=s, v=v, eps=eps,
    )


DENSE = full_pipeline(X, epochs, TIMING['fd_end'], TIMING['grok'])
patterns = {'dense (every 100)': DENSE}

for stride in [500, 1000]:
    mask = (epochs % stride == 0)
    mask[0] = True; mask[-1] = True
    patterns[f'uniform sparse ({stride})'] = full_pipeline(
        X[mask], epochs[mask], TIMING['fd_end'], TIMING['grok']
    )

for cutoff in [3000, 4000, 6000]:
    mask = ((epochs <= cutoff) & (epochs % 100 == 0)) | ((epochs > cutoff) & (epochs % 500 == 0))
    mask[0] = True; mask[-1] = True
    patterns[f'hybrid dense<={cutoff}, sparse 500 after'] = full_pipeline(
        X[mask], epochs[mask], TIMING['fd_end'], TIMING['grok']
    )

print(f'{"pattern":40s}  {"n":>4s}  {"wiggle":>7s}  {"Δwig":>6s}  {"saddle":>7s}  {"Δsad":>6s}  {"L":>6s}  {"ΔL":>6s}')
print('-' * 90)
for name, r in patterns.items():
    dw = r['wiggle'] - DENSE['wiggle']
    ds = r['saddle'] - DENSE['saddle']
    dL = r['L'] - DENSE['L']
    print(f'{name:40s}  {len(r["eps"]):4d}  {r["wiggle"]:7d}  {dw:+6d}  {r["saddle"]:7d}  {ds:+6d}  {r["L"]:6.2f}  {dL:+6.2f}')


In [ ]:
fig = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.08,
                    subplot_titles=['s(t) — dense vs sparse patterns',
                                    'ds/dt — dense vs sparse patterns'])
palette = ['#1f77b4', '#ff7f0e', '#d62728', '#2ca02c', '#9467bd', '#17becf']

for (name, r), color in zip(patterns.items(), palette):
    fig.add_trace(go.Scatter(x=r['eps'], y=r['s'], mode='lines+markers',
                             name=name, line=dict(color=color, width=2),
                             marker=dict(size=4)), row=1, col=1)
    fig.add_trace(go.Scatter(x=r['eps'], y=r['v'], mode='lines+markers',
                             name=name, line=dict(color=color, width=2),
                             marker=dict(size=4), showlegend=False), row=2, col=1)

for row in (1, 2):
    fig.add_vline(x=TIMING['fd_end'], line=dict(color='orange', dash='dot', width=1), row=row, col=1)
    fig.add_vline(x=TIMING['grok'],   line=dict(color='black',  dash='dash', width=1), row=row, col=1)

fig.update_yaxes(title_text='s',     row=1, col=1)
fig.update_yaxes(title_text='ds/dt', row=2, col=1)
fig.update_xaxes(title_text='epoch', row=2, col=1)
fig.update_layout(
    title=f'{label} — checkpointing sensitivity  (baseline = dense every 100 epochs)',
    template='plotly_white', height=750, width=1100,
)
fig.show()

---

## 7. Broadening — p113/s999/ds598 (canon)

User extended dense (every-100) checkpointing through epoch **15000** on p113/s999/ds598. Reference timing for p113: `fd_end=1200`, `sd_onset=9503`, `grok=12448`, `sd_end=12400`. That's grok + ~21% margin and ~+1k past second-descent end — comfortably above the empirical rule from §6.

Predictions if the saddle-transport picture generalizes:
1. Clean sigmoid `s(t)`.
2. `ds/dt` peak in late plateau / early second descent (i.e., **before** grok), echoing p109's ~950-epoch pre-grok crossing.
3. Terminal-robust under truncation.

We refactor the pipeline into `run_saddle_pipeline(variant)` so this and future variants reuse one function.

In [ ]:
def run_saddle_pipeline(variant, terminal=None):
    """Full saddle-transport pipeline for a variant.

    For non-grokkers (`test_loss_threshold_first_epoch` ∈ {-1, None}), the wiggle
    search window falls back to [fd_end, sd_onset] — the structural analog of
    [fd_end, grok], since 'commitment' for these models lives inside / after
    second descent rather than at a grok point.

    `terminal` defaults to the final checkpoint; pass an explicit epoch to study
    alternative landing points (e.g. 'near-landing' for rebounders).
    """
    p = variant.params
    label = f'p{p["prime"]}/s{p["seed"]}/ds{p["data_seed"]}'
    with open(variant.variant_dir / 'variant_summary.json') as f:
        summary = json.load(f)
    grok_raw = summary.get('test_loss_threshold_first_epoch')
    grok = grok_raw if (grok_raw not in (None, -1)) else None
    timing = {
        'fd_end':   summary['first_descent_window']['end_epoch'],
        'sd_onset': summary.get('second_descent_onset_epoch'),
        'sd_end':   summary.get('second_descent_window', {}).get('end_epoch'),
        'grok':     grok,
    }

    epochs, X = load_mlp_param_trajectory(variant)

    loader = ArtifactLoader(str(variant.variant_dir / 'artifacts'))
    pt = loader.load_cross_epoch('parameter_trajectory')
    pt_eps = pt['epochs']
    mlp_proj = pt['mlp__projections']
    pr3, _ = _compute_rolling_trajectory_metrics(mlp_proj, window=10)
    pc3 = mlp_proj[:, 2]

    # Wiggle search window: [fd_end, grok] if grok exists, else [fd_end, sd_onset]
    search_hi = grok if grok is not None else timing['sd_onset']
    in_win = (pt_eps >= timing['fd_end']) & (pt_eps <= search_hi)
    we_pr3 = int(pt_eps[int(np.argmax(np.where(in_win, pr3, -np.inf)))])
    we_pc3 = int(pt_eps[int(np.argmax(np.where(in_win, np.abs(pc3), -np.inf)))])

    if terminal is None:
        terminal = int(epochs[-1])
    s, L, i_w, i_T = compute_transit_projection(X, epochs, we_pr3, terminal)
    v = compute_velocity(s, epochs)
    saddle = int(epochs[int(np.argmax(v))])

    return dict(
        label=label, variant=variant, summary=summary, timing=timing,
        epochs=epochs, X=X,
        pt_eps=pt_eps, mlp_proj=mlp_proj, pr3=pr3, pc3=pc3,
        wiggle_pr3=we_pr3, wiggle_pc3=we_pc3,
        terminal=terminal, s=s, v=v, L=L, saddle=saddle,
        max_dsdt=float(v.max()),
        min_dsdt=float(v.min()),
        s_min=float(s.min()), s_min_ep=int(epochs[int(np.argmin(s))]),
        s_max=float(s.max()), s_max_ep=int(epochs[int(np.argmax(s))]),
    )


variant_113 = family.get_variant(prime=113, seed=999, data_seed=598)
R113 = run_saddle_pipeline(variant_113)

print(f'{R113["label"]}  ({R113["epochs"][0]} → {R113["epochs"][-1]}, n={len(R113["epochs"])})')
print(f'  timing       : {R113["timing"]}')
print(f'  wiggle (PR₃) : ep {R113["wiggle_pr3"]}  (PR₃={R113["pr3"][np.argmax(R113["pr3"])]:.3f} max overall)')
print(f'  wiggle (|PC3|): ep {R113["wiggle_pc3"]}')
print(f'  transit L    : {R113["L"]:.3f}')
print(f'  saddle ep    : {R113["saddle"]}  (max ds/dt = {R113["max_dsdt"]:.2e})')
print(f'  saddle − grok: {R113["saddle"] - R113["timing"]["grok"]:+d} epochs')

In [ ]:
# Wiggle candidates for p113 — same layout as §2 plot for direct comparison
R = R113
fig = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.08,
                    subplot_titles=['MLP trajectory — rolling PR₃',
                                    'MLP trajectory — PC3 coordinate'])
fig.add_trace(go.Scatter(x=R['pt_eps'], y=R['pr3'], mode='lines', name='PR₃',
                         line=dict(color='#d62728', width=2)), row=1, col=1)
fig.add_trace(go.Scatter(x=R['pt_eps'], y=R['pc3'], mode='lines', name='PC3',
                         line=dict(color='#1f77b4', width=2)), row=2, col=1)

for row in (1, 2):
    fig.add_vline(x=R['timing']['fd_end'], line=dict(color='orange', dash='dot', width=1), row=row, col=1)
    fig.add_vline(x=R['timing']['sd_onset'], line=dict(color='teal', dash='dot', width=1), row=row, col=1)
    fig.add_vline(x=R['timing']['grok'],   line=dict(color='black',  dash='dash', width=1), row=row, col=1)
    fig.add_vline(x=R['wiggle_pr3'], line=dict(color='red',    dash='solid', width=1.5), row=row, col=1)
    fig.add_vline(x=R['wiggle_pc3'], line=dict(color='purple', dash='solid', width=1.5), row=row, col=1)

fig.update_yaxes(title_text='PR₃', range=[0.9, 3.1], row=1, col=1)
fig.update_yaxes(title_text='PC3', row=2, col=1)
fig.update_xaxes(title_text='epoch', row=2, col=1)
fig.update_layout(
    title=f'{R["label"]} — wiggle-point candidates<br><sup>orange=fd_end · teal=sd_onset · black=grok · red=PR₃ peak · purple=|PC3| peak</sup>',
    template='plotly_white', height=600, width=1000,
)
fig.show()

In [ ]:
# Sigmoid + velocity for p113 — same layout as §3
R = R113
fig = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.08,
                    subplot_titles=['s(t)  —  scalar projection along transit axis  (0=wiggle, 1=terminal)',
                                    'ds/dt  —  projection velocity'])

fig.add_trace(go.Scatter(x=R['epochs'], y=R['s'], mode='lines+markers', name='s(t)',
                         line=dict(color='#1f77b4', width=2),
                         marker=dict(size=4)), row=1, col=1)
fig.add_hline(y=0.0, line=dict(color='gray', dash='dot', width=1), row=1, col=1)
fig.add_hline(y=1.0, line=dict(color='gray', dash='dot', width=1), row=1, col=1)
fig.add_hline(y=0.5, line=dict(color='rgba(0,0,0,0.15)', dash='dot', width=1), row=1, col=1)

fig.add_trace(go.Scatter(x=R['epochs'], y=R['v'], mode='lines+markers', name='ds/dt',
                         line=dict(color='#d62728', width=2),
                         marker=dict(size=4)), row=2, col=1)

for row in (1, 2):
    fig.add_vline(x=R['timing']['fd_end'],   line=dict(color='orange', dash='dot', width=1), row=row, col=1)
    fig.add_vline(x=R['timing']['sd_onset'], line=dict(color='teal',   dash='dot', width=1), row=row, col=1)
    fig.add_vline(x=R['timing']['grok'],     line=dict(color='black',  dash='dash', width=1), row=row, col=1)
    fig.add_vline(x=R['wiggle_pr3'],         line=dict(color='red',    dash='solid', width=1.5), row=row, col=1)
    fig.add_vline(x=R['saddle'],             line=dict(color='green',  dash='solid', width=1.5), row=row, col=1)

fig.update_yaxes(title_text='s (normalized)', row=1, col=1)
fig.update_yaxes(title_text='ds/dt',         row=2, col=1)
fig.update_xaxes(title_text='epoch', row=2, col=1)
fig.update_layout(
    title=(f'{R["label"]} — transit projection  (wiggle ep={R["wiggle_pr3"]}, terminal ep={R["terminal"]})<br>'
           '<sup>orange=fd_end · teal=sd_onset · black=grok · red=wiggle · green=ds/dt peak</sup>'),
    template='plotly_white', height=700, width=1000, showlegend=False,
)
fig.show()

---

## 8. Falsifier — p101/s485/ds999 (rebounder)

p101/s485/ds999 enters second descent (sd_onset=6463), reaches near-zero test loss at sd_end=9300, then **rebounds out** — final test loss = 0.049. No grok.

Two terminal probes — designed to ask different questions:

- **Terminal = final (24999).** Healthy-pipeline analog. The transit axis points to the rebounded final state. Question: does the trajectory look monotonically sigmoidal toward this terminal, or does it reveal non-monotonic structure?

- **Terminal = near-landing (sd_end=9300).** "Where the model nearly landed." The transit axis points to the bottom of second descent (the model's transient minimum). Question: does the trajectory overshoot 1.0 (= excursion past the near-landing point in MLP-parameter space), and if so, does it return?

The pipeline now falls back to `[fd_end, sd_onset]` for the wiggle search window when no grok exists, so this works without ad-hoc edits.

In [ ]:
variant_101r = family.get_variant(prime=101, seed=485, data_seed=999)

# Pick near-landing epoch from the test-loss minimum inside the second-descent window
md_101r = variant_101r.metadata
test_losses_101r = np.array(md_101r['test_losses'])
with open(variant_101r.variant_dir / 'variant_summary.json') as f:
    summary_101r = json.load(f)
sd_lo = summary_101r['second_descent_window']['start_epoch']
sd_hi = summary_101r['second_descent_window']['end_epoch']
near_land_raw = int(sd_lo + np.argmin(test_losses_101r[sd_lo:sd_hi + 1]))
near_land_loss = float(test_losses_101r[near_land_raw])
final_loss_101r = float(test_losses_101r[-1])

# Snap to nearest checkpoint (dense-100 covers this region)
loader_101r = ArtifactLoader(str(variant_101r.variant_dir / 'artifacts'))
checkpoint_eps = np.array(sorted(loader_101r.get_epochs('parameter_snapshot')))
near_land_ep = int(checkpoint_eps[np.argmin(np.abs(checkpoint_eps - near_land_raw))])

R101_final = run_saddle_pipeline(variant_101r)
R101_land  = run_saddle_pipeline(variant_101r, terminal=near_land_ep)

print(f'p101/s485/ds999 rebounder')
print(f'  test loss: min={near_land_loss:.5f} at ep {near_land_raw} (snapped {near_land_ep})  ·  final={final_loss_101r:.5f}')
print(f'  timing: {R101_final["timing"]}  (grok=None ⇒ wiggle window = [fd_end, sd_onset])')
print(f'  wiggle (PR₃) : ep {R101_final["wiggle_pr3"]}  (PR₃={R101_final["pr3"][np.argmax(R101_final["pr3"])]:.3f} max overall)')
print(f'  wiggle (|PC3|): ep {R101_final["wiggle_pc3"]}')
print()
for tag, R in [('terminal=final', R101_final), (f'terminal=near-land({near_land_ep})', R101_land)]:
    print(f'  --- {tag} ---')
    print(f'    L = {R["L"]:.3f}')
    print(f'    saddle (max ds/dt) = ep {R["saddle"]}  (max ds/dt = {R["max_dsdt"]:.2e}, min ds/dt = {R["min_dsdt"]:.2e})')
    print(f'    s overshoot: max s = {R["s_max"]:+.4f} at ep {R["s_max_ep"]}  (>1.0 ⇒ trajectory passes the terminal)')
    print(f'    s_min pre-wiggle = {R["s_min"]:+.4f} at ep {R["s_min_ep"]}')


In [ ]:
# Wiggle candidates — p101 rebounder
R = R101_final
fig = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.08,
                    subplot_titles=['MLP trajectory — rolling PR₃',
                                    'MLP trajectory — PC3 coordinate'])
fig.add_trace(go.Scatter(x=R['pt_eps'], y=R['pr3'], mode='lines', name='PR₃',
                         line=dict(color='#d62728', width=2)), row=1, col=1)
fig.add_trace(go.Scatter(x=R['pt_eps'], y=R['pc3'], mode='lines', name='PC3',
                         line=dict(color='#1f77b4', width=2)), row=2, col=1)

# No grok line — but mark sd_end (the rebound 'reset' point) in addition to sd_onset
for row in (1, 2):
    fig.add_vline(x=R['timing']['fd_end'],   line=dict(color='orange', dash='dot', width=1), row=row, col=1)
    fig.add_vline(x=R['timing']['sd_onset'], line=dict(color='teal',   dash='dot', width=1), row=row, col=1)
    fig.add_vline(x=R['timing']['sd_end'],   line=dict(color='magenta',dash='dot', width=1), row=row, col=1)
    fig.add_vline(x=R['wiggle_pr3'], line=dict(color='red',    dash='solid', width=1.5), row=row, col=1)
    fig.add_vline(x=R['wiggle_pc3'], line=dict(color='purple', dash='solid', width=1.5), row=row, col=1)

fig.update_yaxes(title_text='PR₃', range=[0.9, 3.1], row=1, col=1)
fig.update_yaxes(title_text='PC3', row=2, col=1)
fig.update_xaxes(title_text='epoch', row=2, col=1)
fig.update_layout(
    title=f'{R["label"]} — wiggle-point candidates (rebounder)<br><sup>orange=fd_end · teal=sd_onset · magenta=sd_end (near-landing) · red=PR₃ peak · purple=|PC3| peak</sup>',
    template='plotly_white', height=600, width=1000,
)
fig.show()

In [ ]:
# Side-by-side: terminal=final vs terminal=near-land. The interesting comparison —
# whether rebound shows as overshoot in the near-land projection.
PROBES = [('terminal=final', R101_final, '#1f77b4'),
          (f'terminal=near-land(ep {near_land_ep})', R101_land, '#d62728')]

fig = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.08,
                    subplot_titles=['s(t) — both terminal probes (s=1.0 line is the near-landing terminal)',
                                    'ds/dt — both terminal probes'])

for tag, R, color in PROBES:
    fig.add_trace(go.Scatter(x=R['epochs'], y=R['s'], mode='lines+markers',
                             name=f'{tag}  (L={R["L"]:.2f})',
                             line=dict(color=color, width=2),
                             marker=dict(size=4)), row=1, col=1)
    fig.add_trace(go.Scatter(x=R['epochs'], y=R['v'], mode='lines+markers',
                             name=tag, line=dict(color=color, width=2),
                             marker=dict(size=4), showlegend=False), row=2, col=1)

# Annotate the s_max overshoot for the near-land probe
R = R101_land
fig.add_annotation(x=R['s_max_ep'], y=R['s_max'], xref='x1', yref='y1',
                   text=f'overshoot: s={R["s_max"]:.3f} at ep {R["s_max_ep"]}',
                   showarrow=True, arrowhead=2, ax=40, ay=-30,
                   bgcolor='rgba(255,255,255,0.85)', bordercolor='#d62728',
                   row=1, col=1)

fig.add_hline(y=0.0, line=dict(color='gray', dash='dot', width=1), row=1, col=1)
fig.add_hline(y=1.0, line=dict(color='gray', dash='dot', width=1), row=1, col=1)

T = R101_final['timing']
for row in (1, 2):
    fig.add_vline(x=T['fd_end'],   line=dict(color='orange',  dash='dot',   width=1), row=row, col=1)
    fig.add_vline(x=T['sd_onset'], line=dict(color='teal',    dash='dot',   width=1), row=row, col=1)
    fig.add_vline(x=T['sd_end'],   line=dict(color='magenta', dash='dot',   width=1), row=row, col=1)
    fig.add_vline(x=R101_final['wiggle_pr3'], line=dict(color='red',  dash='solid', width=1.5), row=row, col=1)

fig.update_yaxes(title_text='s (normalized)', row=1, col=1)
fig.update_yaxes(title_text='ds/dt',         row=2, col=1)
fig.update_xaxes(title_text='epoch', row=2, col=1)
fig.update_layout(
    title=(f'{R101_final["label"]} — rebounder transit projection (two terminals)<br>'
           '<sup>orange=fd_end · teal=sd_onset · magenta=sd_end (near-landing) · red=wiggle</sup>'),
    template='plotly_white', height=750, width=1100,
)
fig.show()

---

## 9. Cross-variant comparison — p109 / p113 / p101-rebounder

Refined dimensionless health-signature candidates (after observing two healthy variants):

- `(saddle − commitment) / commitment` — fractional offset, comparable across timescales
- `|s_min|` — retreat depth pre-wiggle
- `max(ds/dt) × commitment` — grok/commitment-rescaled crossing sharpness

For grokkers, `commitment = grok`. For non-grokkers (rebounders/diverged), `commitment = sd_end` (the near-landing or "where it nearly committed" epoch). This keeps the fractional offset interpretable across variant classes.

Below: full comparison including p101 rebounder with `terminal=final` (apples-to-apples). The rebounder-specific signature — overshoot under `terminal=near-landing` — is in §8, not in this table.

In [ ]:
# Re-bundle p109 results via the same pipeline function so comparison is apples-to-apples
R109 = run_saddle_pipeline(variant)  # variant is p109 from §1
RUNS = [R109, R113, R101_final]      # p101 with terminal=final for parity with healthy variants

# For non-grokkers (p101), use sd_end (near-landing epoch) as the 'commitment' anchor,
# so we can report (saddle - commitment)/commitment as a unified ratio.
def commitment_epoch(R):
    return R['timing']['grok'] if R['timing']['grok'] is not None else R['timing']['sd_end']

def commitment_label(R):
    return 'grok' if R['timing']['grok'] is not None else 'sd_end'

print(f'{"variant":20s}  {"d_mlp":>6s}  {"commit":>10s}  {"wiggle":>7s}  {"saddle":>7s}  '
      f'{"sad-com":>8s}  {"frac":>6s}  {"L":>7s}  {"|s_min|":>8s}  {"max·com":>8s}')
print('-' * 105)
for R in RUNS:
    c = commitment_epoch(R); cl = commitment_label(R)
    sg = R['saddle'] - c
    frac = sg / c
    smin = abs(R['s_min'])
    print(f'{R["label"]:20s}  {R["X"].shape[1]:6d}  {f"{c} ({cl})":>10s}  '
          f'{R["wiggle_pr3"]:7d}  {R["saddle"]:7d}  {sg:+8d}  {frac:+6.2f}  '
          f'{R["L"]:7.2f}  {smin:8.3f}  {R["max_dsdt"]*c:8.2f}')

In [ ]:
# Overlay s(t) and ds/dt on commitment-normalized x-axis (epoch / commitment_epoch)
# - commitment = grok for healthy variants, sd_end for rebounders/non-grokkers
# - lets us compare saddle structure across variants with very different timescales
fig = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.08,
                    subplot_titles=['s(t)  —  normalized by commitment epoch',
                                    'ds/dt × commitment  —  velocity rescaled to commitment-time'])
colors = {'p109/s485/ds598': '#1f77b4',
          'p113/s999/ds598': '#d62728',
          'p101/s485/ds999': '#2ca02c'}

for R in RUNS:
    c = commitment_epoch(R)
    x_norm = R['epochs'] / c
    color = colors[R['label']]
    legend_name = f'{R["label"]} (commit={commitment_label(R)}={c})'
    fig.add_trace(go.Scatter(x=x_norm, y=R['s'], mode='lines+markers',
                             name=legend_name, line=dict(color=color, width=2),
                             marker=dict(size=3)), row=1, col=1)
    fig.add_trace(go.Scatter(x=x_norm, y=R['v'] * c, mode='lines+markers',
                             name=R['label'], line=dict(color=color, width=2),
                             marker=dict(size=3), showlegend=False), row=2, col=1)
    fig.add_vline(x=R['wiggle_pr3'] / c,
                  line=dict(color=color, dash='dot', width=1), row=1, col=1, opacity=0.4)
    fig.add_vline(x=R['saddle'] / c,
                  line=dict(color=color, dash='solid', width=1), row=2, col=1, opacity=0.4)

for row in (1, 2):
    fig.add_vline(x=1.0, line=dict(color='black', dash='dash', width=1), row=row, col=1)

fig.add_hline(y=0.0, line=dict(color='gray', dash='dot', width=1), row=1, col=1)
fig.add_hline(y=1.0, line=dict(color='gray', dash='dot', width=1), row=1, col=1)
fig.add_hline(y=0.5, line=dict(color='rgba(0,0,0,0.15)', dash='dot', width=1), row=1, col=1)

fig.update_yaxes(title_text='s (normalized)',  row=1, col=1)
fig.update_yaxes(title_text='ds/dt × commit',  row=2, col=1)
fig.update_xaxes(title_text='epoch / commitment_epoch', row=2, col=1)
fig.update_layout(
    title='p109 / p113 / p101-rebounder — transit projection on commitment-normalized time<br>'
          '<sup>black dash = commitment (x=1) · colored dots = each run\'s wiggle (top) and saddle (bottom)</sup>',
    template='plotly_white', height=750, width=1100,
)
fig.show()

---

## 10. Stress test — p59/s485/ds598 (overshooter, late-grokker)

Dense every-100 checkpointing across the full 34999-epoch run. Structural anomalies for this variant:

- **Overshoot spike.** `test_loss_max = 31.36 at ep 1441` — right after fd_end (ep 1000). Creates a sharp, early PR₃ spike that canon doesn't have.
- **No-grok-by-threshold, but test loss actually goes to zero.** `test_loss_threshold_first_epoch = -1`, yet `test_loss_min = 2.24e-06 at ep 30885 ≈ sd_end`. Pipeline treats it as non-grokker; commitment anchor = sd_end = 30900.
- **Irregular cycling that may not terminate at sd_end** (eyeballed from `dimensionality.timeseries`). Breaks canon's "cycling stops at sd_end" regularity — canon shows 3 clean peaks + flat tail; p59 shows ~4–5 irregular peaks with no clean flat tail.

Three predictions this variant lets us test:

1. **Wiggle-peak ambiguity.** PR₃ in `[fd_end=1000, sd_onset=15553]` has multiple local maxima. The argmax-based wiggle is fragile; we enumerate all peaks with `scipy.signal.find_peaks` and overlay the resulting sigmoids to see how much shape depends on which peak we anchor to.
2. **Terminal sensitivity.** Canon in §5 had ~6% L variation across terminals. If cycling persists past sd_end, p59's transit axis under `terminal=final` vs `terminal=sd_end` vs `terminal=sd_onset` should differ substantively.
3. **Shape non-monotonicity.** The model may retreat/advance multiple times. If so, `s(t)` won't be cleanly sigmoidal.


In [ ]:
from scipy.signal import find_peaks

variant_59 = family.get_variant(prime=59, seed=485, data_seed=598)
R59 = run_saddle_pipeline(variant_59)

# Enumerate all prominent PR₃ peaks in the wiggle search window
search_lo = R59['timing']['fd_end']
search_hi = R59['timing']['sd_onset']  # grok is None → falls back to sd_onset
in_win = (R59['pt_eps'] >= search_lo) & (R59['pt_eps'] <= search_hi)
pr3_win = R59['pr3'][in_win]
eps_win = R59['pt_eps'][in_win]

peak_idx, props = find_peaks(pr3_win, prominence=0.05)
peak_epochs = eps_win[peak_idx].astype(int).tolist()
peak_pr3 = pr3_win[peak_idx].tolist()
peak_prominence = props['prominences'].tolist()

print(f'{R59["label"]}  (n={len(R59["epochs"])}, epoch range {R59["epochs"][0]} → {R59["epochs"][-1]})')
print(f'  timing            : {R59["timing"]}')
print(f'  test_loss_min     : 2.24e-06 at ep 30885 (effectively coincident with sd_end={R59["timing"]["sd_end"]})')
print()
print(f'  --- PR₃ peaks in wiggle search window [{search_lo}, {search_hi}]  (prominence ≥ 0.05) ---')
for ep, pr, prom in sorted(zip(peak_epochs, peak_pr3, peak_prominence), key=lambda x: -x[2]):
    marker = '  ← argmax (pipeline wiggle)' if ep == R59['wiggle_pr3'] else ''
    print(f'    ep {ep:5d}  PR₃={pr:.3f}  prominence={prom:.3f}{marker}')
print()
print(f'  pipeline wiggle (argmax)            : ep {R59["wiggle_pr3"]}')
print(f'  pipeline saddle (ds/dt max, T=final): ep {R59["saddle"]}')
print(f'  transit L (terminal=final)          : {R59["L"]:.3f}')
print(f'  s_min pre-wiggle                    : {R59["s_min"]:+.4f} at ep {R59["s_min_ep"]}')
print(f'  s_max                               : {R59["s_max"]:+.4f} at ep {R59["s_max_ep"]}')


In [ ]:
# PR₃ trajectory + PC3 + all enumerated peaks — visual stress test
fig = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.08,
                    subplot_titles=['MLP trajectory — rolling PR₃  (all prominent peaks in [fd_end, sd_onset])',
                                    'MLP trajectory — PC3 coordinate'])
fig.add_trace(go.Scatter(x=R59['pt_eps'], y=R59['pr3'], mode='lines', name='PR₃',
                         line=dict(color='#d62728', width=2)), row=1, col=1)
fig.add_trace(go.Scatter(x=R59['pt_eps'], y=R59['pc3'], mode='lines', name='PC3',
                         line=dict(color='#1f77b4', width=2)), row=2, col=1)

peak_palette = ['#2ca02c', '#ff7f0e', '#9467bd', '#8c564b', '#e377c2', '#17becf']
for i, (ep, pr) in enumerate(sorted(zip(peak_epochs, peak_pr3))):
    color = peak_palette[i % len(peak_palette)]
    width = 2.0 if ep == R59['wiggle_pr3'] else 1.0
    fig.add_vline(x=ep, line=dict(color=color, dash='solid', width=width), row=1, col=1)
    fig.add_annotation(x=ep, y=pr, text=f'ep {ep}<br>PR₃={pr:.2f}',
                       showarrow=False, yshift=12,
                       bgcolor='rgba(255,255,255,0.85)',
                       font=dict(size=9, color=color), row=1, col=1)

for row in (1, 2):
    fig.add_vline(x=R59['timing']['fd_end'],   line=dict(color='orange',  dash='dot', width=1), row=row, col=1)
    fig.add_vline(x=R59['timing']['sd_onset'], line=dict(color='teal',    dash='dot', width=1), row=row, col=1)
    fig.add_vline(x=R59['timing']['sd_end'],   line=dict(color='magenta', dash='dot', width=1), row=row, col=1)

fig.update_yaxes(title_text='PR₃', row=1, col=1)
fig.update_yaxes(title_text='PC3', row=2, col=1)
fig.update_xaxes(title_text='epoch', row=2, col=1)
fig.update_layout(
    title=f'{R59["label"]} — wiggle-peak enumeration (overshooter, late-grokker)<br>'
          '<sup>orange=fd_end · teal=sd_onset · magenta=sd_end · colored lines = PR₃ peaks with prominence ≥ 0.05</sup>',
    template='plotly_white', height=600, width=1100, showlegend=False,
)
fig.show()


In [ ]:
# Sigmoid per wiggle-peak choice — how much does shape depend on anchor?
fig = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.08,
                    subplot_titles=['s(t) per wiggle-peak choice  (terminal = final = 34999)',
                                    'ds/dt per wiggle-peak choice'])

terminal_final = int(R59['epochs'][-1])

for i, (we, pr) in enumerate(sorted(zip(peak_epochs, peak_pr3))):
    color = peak_palette[i % len(peak_palette)]
    s_c, L_c, _, _ = compute_transit_projection(R59['X'], R59['epochs'], we, terminal_final)
    v_c = compute_velocity(s_c, R59['epochs'])
    saddle_c = int(R59['epochs'][int(np.argmax(v_c))])
    legend = f'wiggle ep {we}  (PR₃={pr:.2f}, L={L_c:.2f}, saddle ep {saddle_c}, s_min={s_c.min():+.2f})'
    fig.add_trace(go.Scatter(x=R59['epochs'], y=s_c, mode='lines',
                             name=legend, line=dict(color=color, width=2)), row=1, col=1)
    fig.add_trace(go.Scatter(x=R59['epochs'], y=v_c, mode='lines', name=legend,
                             line=dict(color=color, width=2), showlegend=False), row=2, col=1)

fig.add_hline(y=0.0, line=dict(color='gray', dash='dot', width=1), row=1, col=1)
fig.add_hline(y=1.0, line=dict(color='gray', dash='dot', width=1), row=1, col=1)

for row in (1, 2):
    fig.add_vline(x=R59['timing']['fd_end'],   line=dict(color='orange',  dash='dot', width=1), row=row, col=1)
    fig.add_vline(x=R59['timing']['sd_onset'], line=dict(color='teal',    dash='dot', width=1), row=row, col=1)
    fig.add_vline(x=R59['timing']['sd_end'],   line=dict(color='magenta', dash='dot', width=1), row=row, col=1)

fig.update_yaxes(title_text='s (normalized)', row=1, col=1)
fig.update_yaxes(title_text='ds/dt', row=2, col=1)
fig.update_xaxes(title_text='epoch', row=2, col=1)
fig.update_layout(
    title=f'{R59["label"]} — wiggle-choice sensitivity  (every prominent PR₃ peak as anchor)<br>'
          '<sup>orange=fd_end · teal=sd_onset · magenta=sd_end</sup>',
    template='plotly_white', height=750, width=1100,
)
fig.show()


In [ ]:
# Terminal sensitivity — pipeline's PR₃-argmax wiggle, vary terminal across the run
TERMINAL_SET_59 = [
    ('terminal=sd_onset', R59['timing']['sd_onset'], '#2ca02c'),
    ('terminal=sd_end',   R59['timing']['sd_end'],   '#ff7f0e'),
    ('terminal=final',    int(R59['epochs'][-1]),    '#d62728'),
]

fig = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.08,
                    subplot_titles=['s(t) per terminal choice  (wiggle = PR₃ argmax)',
                                    'ds/dt per terminal choice'])

for (name, te, color) in TERMINAL_SET_59:
    te_snap = int(R59['epochs'][np.argmin(np.abs(R59['epochs'] - te))])
    s_c, L_c, _, _ = compute_transit_projection(R59['X'], R59['epochs'], R59['wiggle_pr3'], te_snap)
    v_c = compute_velocity(s_c, R59['epochs'])
    saddle_c = int(R59['epochs'][int(np.argmax(v_c))])
    fig.add_trace(go.Scatter(x=R59['epochs'], y=s_c, mode='lines',
                             name=f'{name} (ep {te_snap}, L={L_c:.2f}, saddle ep {saddle_c})',
                             line=dict(color=color, width=2)), row=1, col=1)
    fig.add_trace(go.Scatter(x=R59['epochs'], y=v_c, mode='lines', name=name,
                             line=dict(color=color, width=2), showlegend=False), row=2, col=1)

fig.add_hline(y=0.0, line=dict(color='gray', dash='dot', width=1), row=1, col=1)
fig.add_hline(y=1.0, line=dict(color='gray', dash='dot', width=1), row=1, col=1)

for row in (1, 2):
    fig.add_vline(x=R59['timing']['fd_end'],   line=dict(color='orange',  dash='dot', width=1), row=row, col=1)
    fig.add_vline(x=R59['timing']['sd_onset'], line=dict(color='teal',    dash='dot', width=1), row=row, col=1)
    fig.add_vline(x=R59['timing']['sd_end'],   line=dict(color='magenta', dash='dot', width=1), row=row, col=1)
    fig.add_vline(x=R59['wiggle_pr3'],         line=dict(color='red',     dash='solid', width=1.5), row=row, col=1)

fig.update_yaxes(title_text='s (normalized)', row=1, col=1)
fig.update_yaxes(title_text='ds/dt', row=2, col=1)
fig.update_xaxes(title_text='epoch', row=2, col=1)
fig.update_layout(
    title=(f'{R59["label"]} — terminal sensitivity  (wiggle ep={R59["wiggle_pr3"]})<br>'
           '<sup>Canon (§5) had ~6% L variation across terminals — compare here.  orange=fd_end · teal=sd_onset · magenta=sd_end · red=wiggle</sup>'),
    template='plotly_white', height=750, width=1100,
)
fig.show()


---

## 11. Cross-variant comparison (updated) — p109 / p113 / p101-rebounder / p59-overshooter

Add p59 to the dimensionless signature table and commitment-normalized overlay. Commitment anchor for p59 = `sd_end = 30900` (pipeline treats it as non-grokker since `test_loss_threshold_first_epoch = -1`, even though test_loss reaches 2.24e-06 at ep 30885 ≈ sd_end).

What to watch for in the overlay:

- **Is the p59 sigmoid shape roughly rescalable to canon's?** If `(saddle − commit)/commit ≈ −0.16 to −0.18`, the fractional-offset signature holds across a fourth variant.
- **Where does p59's `|s_min|` land?** p109 = 0.27, p113 = 0.08, p101 (terminal=final) = 0.15. For an overshooter, we might expect a large retreat depth — or the opposite, depending on whether "overshoot" means excursion in test loss vs in parameter space.
- **Is the sigmoid monotonic on commitment-normalized time?** Non-monotonicity here would be the qualitative falsifier.


In [ ]:
# Apples-to-apples table + commitment-normalized overlay including p59
RUNS_EXT = [R109, R113, R101_final, R59]
variant_class = {
    'p109/s485/ds598': 'healthy',
    'p113/s999/ds598': 'healthy canon',
    'p101/s485/ds999': 'rebounder',
    'p59/s485/ds598':  'overshooter/late',
}
colors_ext = {
    'p109/s485/ds598': '#1f77b4',
    'p113/s999/ds598': '#d62728',
    'p101/s485/ds999': '#2ca02c',
    'p59/s485/ds598':  '#9467bd',
}

print(f'{"variant":20s}  {"class":18s}  {"d_mlp":>6s}  {"commit":>13s}  {"wiggle":>7s}  {"saddle":>7s}  '
      f'{"sad-com":>8s}  {"frac":>6s}  {"L":>7s}  {"|s_min|":>8s}  {"max·com":>8s}')
print('-' * 130)
for R in RUNS_EXT:
    c = commitment_epoch(R); cl = commitment_label(R)
    sg = R['saddle'] - c
    frac = sg / c
    smin = abs(R['s_min'])
    cls = variant_class.get(R['label'], '?')
    print(f'{R["label"]:20s}  {cls:18s}  {R["X"].shape[1]:6d}  {f"{c} ({cl})":>13s}  '
          f'{R["wiggle_pr3"]:7d}  {R["saddle"]:7d}  {sg:+8d}  {frac:+6.2f}  '
          f'{R["L"]:7.2f}  {smin:8.3f}  {R["max_dsdt"]*c:8.2f}')

fig = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.08,
                    subplot_titles=['s(t)  —  normalized by commitment epoch',
                                    'ds/dt × commitment  —  velocity rescaled to commitment-time'])

for R in RUNS_EXT:
    c = commitment_epoch(R)
    x_norm = R['epochs'] / c
    color = colors_ext[R['label']]
    legend_name = f'{R["label"]} ({variant_class[R["label"]]}, commit={commitment_label(R)}={c})'
    fig.add_trace(go.Scatter(x=x_norm, y=R['s'], mode='lines+markers',
                             name=legend_name, line=dict(color=color, width=2),
                             marker=dict(size=3)), row=1, col=1)
    fig.add_trace(go.Scatter(x=x_norm, y=R['v'] * c, mode='lines+markers',
                             name=R['label'], line=dict(color=color, width=2),
                             marker=dict(size=3), showlegend=False), row=2, col=1)
    fig.add_vline(x=R['wiggle_pr3'] / c,
                  line=dict(color=color, dash='dot', width=1), row=1, col=1, opacity=0.4)
    fig.add_vline(x=R['saddle'] / c,
                  line=dict(color=color, dash='solid', width=1), row=2, col=1, opacity=0.4)

for row in (1, 2):
    fig.add_vline(x=1.0, line=dict(color='black', dash='dash', width=1), row=row, col=1)

fig.add_hline(y=0.0, line=dict(color='gray', dash='dot', width=1), row=1, col=1)
fig.add_hline(y=1.0, line=dict(color='gray', dash='dot', width=1), row=1, col=1)
fig.add_hline(y=0.5, line=dict(color='rgba(0,0,0,0.15)', dash='dot', width=1), row=1, col=1)

fig.update_yaxes(title_text='s (normalized)',  row=1, col=1)
fig.update_yaxes(title_text='ds/dt × commit',  row=2, col=1)
fig.update_xaxes(title_text='epoch / commitment_epoch', row=2, col=1)
fig.update_layout(
    title='Four-variant overlay on commitment-normalized time<br>'
          '<sup>black dash = commitment (x=1) · colored dots = each run\'s wiggle (top) and saddle (bottom)</sup>',
    template='plotly_white', height=750, width=1100,
)
fig.show()


---

## 12. On-arc vs off-arc decomposition — why cyclic PR₃ can coexist with a smooth global arc

**Hypothesis.** Rolling-PR₃ peaks come in two flavors:

1. **Arc-curvature peaks.** The global arc itself is bending in the window → local motion spans multiple directions because the *arc tangent* is rotating. These align with the saddle region visible in PC2/PC3 scatter.
2. **Off-arc (wobble) peaks.** Small fluctuations perpendicular to a locally straight arc transiently dominate window variance. These have no correspondence in the global trajectory.

**Diagnostic.** At each rolling-window end t:
- `τ(t)` — long-range tangent (smoothed velocity over ±20 checkpoints). Represents the slow arc direction.
- `u₁(t)` — local PC1 of the 10-checkpoint window. Represents the dominant local motion direction.
- `alignment(t) = |u₁ · τ|` — cos-angle between them.

`alignment ≈ 1` at a PR₃ peak ⇒ on-arc (arc is curving, local motion tracks tangent rotation).
`alignment ≈ 0` at a PR₃ peak ⇒ off-arc (motion is dominated by wobble orthogonal to arc).

All four variants' PR₃ and `mlp_proj` traces are already in `RUNS_EXT`; we just need to add the alignment signal.

In [ ]:
def compute_arc_decomposition(proj, epochs, smooth_k=20, window=10):
    """Compute local-PC1/arc-tangent alignment along a projected trajectory.

    proj      : (n_epochs, k) projected MLP trajectory (matches _compute_rolling_trajectory_metrics domain)
    smooth_k  : half-window for tangent smoothing — should be >> `window`
    window    : rolling PCA window (matches pipeline's rolling PR₃ window)
    Returns dict with tau, curvature, local_pc1, alignment (all length n_epochs).
    """
    proj = np.asarray(proj, dtype=np.float64)
    epochs = np.asarray(epochs, dtype=np.float64)
    n, k = proj.shape

    tau = np.zeros((n, k))
    for t in range(n):
        lo = max(0, t - smooth_k)
        hi = min(n - 1, t + smooth_k)
        delta = proj[hi] - proj[lo]
        norm = np.linalg.norm(delta)
        tau[t] = delta / norm if norm > 1e-12 else 0.0

    curvature = np.zeros(n)
    for t in range(1, n - 1):
        dt = epochs[t + 1] - epochs[t - 1]
        if dt > 0:
            curvature[t] = np.linalg.norm(tau[t + 1] - tau[t - 1]) / dt

    local_pc1 = np.zeros((n, k))
    alignment = np.full(n, np.nan)
    for t in range(window - 1, n):
        w = proj[t - window + 1 : t + 1]
        wc = w - w.mean(axis=0)
        _, _, vt = np.linalg.svd(wc, full_matrices=False)
        u1 = vt[0]
        local_pc1[t] = u1
        alignment[t] = abs(float(np.dot(u1, tau[t])))

    return dict(tau=tau, curvature=curvature, local_pc1=local_pc1, alignment=alignment)


DECOMP = {}
for R in RUNS_EXT:
    d = compute_arc_decomposition(R['mlp_proj'], R['pt_eps'], smooth_k=20, window=10)
    DECOMP[R['label']] = d
    # Summary: alignment at each PR₃ peak in [fd_end, commit]
    search_lo = R['timing']['fd_end']
    search_hi = commitment_epoch(R)
    in_win = (R['pt_eps'] >= search_lo) & (R['pt_eps'] <= search_hi)
    pr3_win = R['pr3'][in_win]
    eps_win = R['pt_eps'][in_win]
    align_win = d['alignment'][in_win]
    peaks_idx, props = find_peaks(pr3_win, prominence=0.05)
    print(f'{R["label"]:20s}  ({variant_class[R["label"]]})')
    if len(peaks_idx) == 0:
        print(f'    no peaks with prominence ≥ 0.05 in [{search_lo}, {search_hi}]')
    else:
        for i, pk in enumerate(peaks_idx):
            ep = int(eps_win[pk])
            pr3_val = float(pr3_win[pk])
            align_val = float(align_win[pk])
            flavor = 'on-arc' if align_val > 0.7 else ('off-arc' if align_val < 0.4 else 'mixed')
            print(f'    ep {ep:5d}  PR₃={pr3_val:.3f}  alignment={align_val:.3f}  → {flavor}')
    print()


In [ ]:
# Per-variant overlay — PR₃ trace with alignment color-coded along the line
from plotly.colors import sample_colorscale

fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=[f'{R["label"]}  ({variant_class[R["label"]]})' for R in RUNS_EXT],
    shared_yaxes=False, vertical_spacing=0.12, horizontal_spacing=0.08,
)

for i, R in enumerate(RUNS_EXT):
    r, c = divmod(i, 2)
    r, c = r + 1, c + 1
    d = DECOMP[R['label']]
    eps = R['pt_eps']
    pr3 = R['pr3']
    align = d['alignment']
    commit = commitment_epoch(R)
    finite = np.isfinite(align)

    # PR₃ trace (grey) — full trace
    fig.add_trace(go.Scatter(
        x=eps, y=pr3, mode='lines', line=dict(color='rgba(120,120,120,0.6)', width=1.2),
        name='PR₃', showlegend=(i == 0), legendgroup='pr3',
    ), row=r, col=c)

    # Alignment-colored scatter overlay — only where alignment is defined
    eps_f = eps[finite]
    pr3_f = pr3[finite]
    align_f = np.clip(align[finite], 0, 1)
    colors = sample_colorscale('RdBu_r', align_f)
    fig.add_trace(go.Scatter(
        x=eps_f, y=pr3_f, mode='markers',
        marker=dict(color=colors, size=4, line=dict(width=0),
                    showscale=(i == 0),
                    colorscale='RdBu_r',
                    cmin=0, cmax=1,
                    colorbar=dict(title='alignment<br>|u₁·τ|', len=0.8, y=0.5, x=1.05) if i == 0 else None),
        name='align', showlegend=False, hovertemplate='ep %{x}<br>PR₃ %{y:.3f}<extra></extra>',
    ), row=r, col=c)

    # Peak markers (restricted to wiggle window, same as §12 classification)
    search_lo = R['timing']['fd_end']
    in_win = (eps >= search_lo) & (eps <= commit)
    pr3_win = pr3[in_win]
    eps_win = eps[in_win]
    peaks_idx, _ = find_peaks(pr3_win, prominence=0.05)
    if len(peaks_idx):
        fig.add_trace(go.Scatter(
            x=eps_win[peaks_idx], y=pr3_win[peaks_idx], mode='markers',
            marker=dict(symbol='diamond', size=11, color='black', line=dict(width=1, color='white')),
            name='peak', showlegend=(i == 0), legendgroup='peak',
            hovertemplate='ep %{x}<br>PR₃ %{y:.3f}<extra></extra>',
        ), row=r, col=c)

    fig.add_vline(x=commit, line=dict(color='green', width=1, dash='dot'), row=r, col=c)

    fig.update_xaxes(title_text='epoch', row=r, col=c)
    fig.update_yaxes(title_text='PR₃', range=[0.95, 3.05], row=r, col=c)

fig.update_layout(height=720, width=1100, title_text='PR₃ peaks colored by arc-alignment — on-arc (red) vs off-arc (blue)')
fig.show()


In [ ]:
# Summary scatter: does peak height sort with alignment? — one point per PR₃ peak, all variants
fig_s = go.Figure()
colors_ext_map = colors_ext

for R in RUNS_EXT:
    d = DECOMP[R['label']]
    search_lo = R['timing']['fd_end']
    search_hi = commitment_epoch(R)
    in_win = (R['pt_eps'] >= search_lo) & (R['pt_eps'] <= search_hi)
    pr3_win = R['pr3'][in_win]
    align_win = d['alignment'][in_win]
    peaks_idx, _ = find_peaks(pr3_win, prominence=0.05)
    if len(peaks_idx) == 0:
        continue
    fig_s.add_trace(go.Scatter(
        x=align_win[peaks_idx],
        y=pr3_win[peaks_idx],
        mode='markers',
        marker=dict(size=13, color=colors_ext_map[R['label']], line=dict(width=1, color='white')),
        name=f'{R["label"]} ({variant_class[R["label"]]})',
        hovertemplate=f'{R["label"]}<br>align %{{x:.3f}}<br>PR₃ %{{y:.3f}}<extra></extra>',
    ))

fig_s.add_vline(x=0.4, line=dict(color='grey', width=1, dash='dot'))
fig_s.add_vline(x=0.7, line=dict(color='grey', width=1, dash='dot'))
fig_s.add_annotation(x=0.2, y=3.0, text='off-arc<br>(wobble)', showarrow=False, font=dict(color='steelblue'))
fig_s.add_annotation(x=0.85, y=3.0, text='on-arc<br>(curvature)', showarrow=False, font=dict(color='firebrick'))

fig_s.update_layout(
    title='PR₃ peaks — height vs arc-alignment',
    xaxis=dict(title='alignment  |u₁ · τ|', range=[0, 1.02]),
    yaxis=dict(title='PR₃ at peak', range=[0.95, 3.05]),
    height=480, width=760,
)
fig_s.show()


---

## 13. p101/s999/ds598 — long-bumpy descent, cross-check against manual anchors

Second candidate data point for the serial-saddle reframe. This variant was dense-checkpointed after §12; the manual notebook (`saddle_transport_manual.ipynb`) places its anchors at wiggle=2100, terminal=29200, with a visible **two-event transit**: early wiggle-recovery + second ds/dt peak near grok (~13-14k).

Predictions this section tests:
1. Pipeline `argmax(ds/dt)` in [fd_end, grok] picks the early wiggle-recovery peak, not the late commitment event (same failure mode as p59's overshoot masking — here the first peak isn't an overshoot but dominates anyway).
2. §12 decomposition finds 1-2 on-arc peaks, with the second near grok ⇒ correspondence with manual 2nd-event picture; p101/s999/ds598 is a **middle case** not a serial-saddle like p59.
3. If instead §12 finds 3+ on-arc peaks ⇒ serial-saddle is a spectrum, p101/s999/ds598 sits partway between canon and p59.

In [ ]:
# Load p101/s999/ds598 and run algorithmic pipeline
variant_p101_999 = family.get_variant(prime=101, seed=999, data_seed=598)
R_p101_999 = run_saddle_pipeline(variant_p101_999)

print(f'p101/s999/ds598  (long bumpy descent)')
print(f'  n_epochs         : {len(R_p101_999["epochs"])}')
print(f'  epoch range      : {R_p101_999["epochs"][0]} → {R_p101_999["epochs"][-1]}')
print(f'  timing           : {R_p101_999["timing"]}')
print(f'  pipeline wiggle  : ep {R_p101_999["wiggle_pr3"]} (argmax rolling PR₃)')
print(f'  pipeline saddle  : ep {R_p101_999["saddle"]} (argmax ds/dt, terminal=final)')
print(f'  transit L        : {R_p101_999["L"]:.3f}')
print()

# Manual anchors from user's saddle_transport_manual notebook
MANUAL_WIGGLE_P101_999 = 2100
MANUAL_TERMINAL_P101_999 = 29200

eps_p = R_p101_999['epochs']
X_p = R_p101_999['X']

# Snap to nearest available checkpoint if exact match unavailable
i_manual_w = int(np.argmin(np.abs(eps_p - MANUAL_WIGGLE_P101_999)))
i_manual_T = int(np.argmin(np.abs(eps_p - MANUAL_TERMINAL_P101_999)))
manual_wiggle = int(eps_p[i_manual_w])
manual_terminal = int(eps_p[i_manual_T])

s_manual, L_manual, iw_m, iT_m = compute_transit_projection(X_p, eps_p, manual_wiggle, manual_terminal)
v_manual = compute_velocity(s_manual, eps_p)

# Algorithmic transit with terminal=grok (alternative anchor)
grok_p = R_p101_999['timing']['grok']
if grok_p is not None:
    s_grok, L_grok, _, _ = compute_transit_projection(X_p, eps_p, R_p101_999['wiggle_pr3'], grok_p)
    v_grok = compute_velocity(s_grok, eps_p)
else:
    s_grok, v_grok = None, None

print(f'  manual wiggle    : ep {manual_wiggle}')
print(f'  manual terminal  : ep {manual_terminal}')
print(f'  manual transit L : {L_manual:.3f}')

# Find ds/dt peaks in manual projection, restricted to [fd_end, manual_terminal]
fd_end = R_p101_999['timing']['fd_end']
in_post_fd = (eps_p > fd_end) & (eps_p <= manual_terminal)
v_search = np.where(in_post_fd, v_manual, -np.inf)
manual_peaks_idx, manual_props = find_peaks(
    np.where(in_post_fd, v_manual, 0),
    prominence=v_manual[in_post_fd].max() * 0.1,
)
manual_peak_eps = [int(eps_p[i]) for i in manual_peaks_idx]
manual_peak_vals = [float(v_manual[i]) for i in manual_peaks_idx]
print(f'  manual ds/dt peaks in [{fd_end}, {manual_terminal}]:')
for ep, val in zip(manual_peak_eps, manual_peak_vals):
    print(f'    ep {ep:5d}  ds/dt={val:.3e}')


In [ ]:
# §12 decomposition on p101/s999/ds598
decomp_p101_999 = compute_arc_decomposition(R_p101_999['mlp_proj'], R_p101_999['pt_eps'], smooth_k=20, window=10)
DECOMP[R_p101_999['label']] = decomp_p101_999

# Classify PR₃ peaks in [fd_end, commit]
search_lo = R_p101_999['timing']['fd_end']
search_hi = commitment_epoch(R_p101_999)
in_win = (R_p101_999['pt_eps'] >= search_lo) & (R_p101_999['pt_eps'] <= search_hi)
pr3_win = R_p101_999['pr3'][in_win]
eps_win = R_p101_999['pt_eps'][in_win]
align_win = decomp_p101_999['alignment'][in_win]

peaks_idx, _ = find_peaks(pr3_win, prominence=0.05)

print(f'{R_p101_999["label"]}  (long bumpy descent)   wiggle window [{search_lo}, {search_hi}]')
if len(peaks_idx) == 0:
    print(f'    no PR₃ peaks with prominence ≥ 0.05')
else:
    for pk in peaks_idx:
        ep = int(eps_win[pk])
        pr3_val = float(pr3_win[pk])
        align_val = float(align_win[pk])
        flavor = 'on-arc' if align_val > 0.7 else ('off-arc' if align_val < 0.4 else 'mixed')
        # Check whether this on-arc peak is near grok (user's 2nd-event window)
        near_grok = ''
        if grok_p is not None and flavor == 'on-arc':
            if abs(ep - grok_p) < 3000:
                near_grok = '  ← near grok (matches user 2nd ds/dt event)'
        print(f'    ep {ep:5d}  PR₃={pr3_val:.3f}  alignment={align_val:.3f}  → {flavor}{near_grok}')

print()
on_arc_count = sum(1 for pk in peaks_idx if align_win[pk] > 0.7)
print(f'  on-arc peak count in [fd_end, commit]: {on_arc_count}')
print(f'  comparison table (see §12 results):')
print(f'    p113 canon         : 1 on-arc')
print(f'    p101/s485/ds999 reb: 1 on-arc')
print(f'    p59 super-critical : 4 on-arc')
print(f'    p101/s999/ds598    : {on_arc_count} on-arc  ← this run')


In [ ]:
# 3-panel cross-check: manual transit, algorithmic transit (terminal=grok), and §12 decomposition
fig = make_subplots(
    rows=3, cols=1, shared_xaxes=True, vertical_spacing=0.06,
    subplot_titles=[
        f's(t) — manual chord (wiggle={manual_wiggle}, terminal={manual_terminal})  vs  algorithmic chord (terminal=grok={grok_p})',
        'ds/dt — projection velocity for both chords',
        f'§12 rolling PR₃ colored by arc-alignment  |u₁·τ|',
    ],
    row_heights=[0.33, 0.33, 0.34],
)

# Row 1: s(t) for both chords
fig.add_trace(go.Scatter(
    x=eps_p, y=s_manual, mode='lines', line=dict(color='#1f77b4', width=2),
    name='s(t) manual', hovertemplate='manual<br>ep %{x}<br>s %{y:.3f}<extra></extra>',
), row=1, col=1)
if s_grok is not None:
    fig.add_trace(go.Scatter(
        x=eps_p, y=s_grok, mode='lines', line=dict(color='#d62728', width=2, dash='dash'),
        name='s(t) algorithmic (terminal=grok)', hovertemplate='algo<br>ep %{x}<br>s %{y:.3f}<extra></extra>',
    ), row=1, col=1)
fig.add_hline(y=0, line=dict(color='grey', width=1, dash='dot'), row=1, col=1)
fig.add_hline(y=1, line=dict(color='grey', width=1, dash='dot'), row=1, col=1)

# Row 2: ds/dt for both chords
fig.add_trace(go.Scatter(
    x=eps_p, y=v_manual, mode='lines', line=dict(color='#1f77b4', width=1.5),
    name='ds/dt manual', showlegend=False,
), row=2, col=1)
if v_grok is not None:
    fig.add_trace(go.Scatter(
        x=eps_p, y=v_grok, mode='lines', line=dict(color='#d62728', width=1.5, dash='dash'),
        name='ds/dt algo', showlegend=False,
    ), row=2, col=1)
# Mark manual ds/dt peaks
if manual_peak_eps:
    fig.add_trace(go.Scatter(
        x=manual_peak_eps, y=manual_peak_vals, mode='markers',
        marker=dict(symbol='diamond', size=13, color='#1f77b4', line=dict(width=1.5, color='white')),
        name='manual ds/dt peaks',
    ), row=2, col=1)

# Row 3: PR₃ colored by alignment + peaks
from plotly.colors import sample_colorscale
pr3_full = R_p101_999['pr3']
eps_full = R_p101_999['pt_eps']
align_full = decomp_p101_999['alignment']
finite = np.isfinite(align_full)

fig.add_trace(go.Scatter(
    x=eps_full, y=pr3_full, mode='lines',
    line=dict(color='rgba(120,120,120,0.5)', width=1.2),
    name='PR₃', showlegend=False,
), row=3, col=1)
colors = sample_colorscale('RdBu_r', np.clip(align_full[finite], 0, 1))
fig.add_trace(go.Scatter(
    x=eps_full[finite], y=pr3_full[finite], mode='markers',
    marker=dict(color=colors, size=4, line=dict(width=0),
                showscale=True, colorscale='RdBu_r', cmin=0, cmax=1,
                colorbar=dict(title='align<br>|u₁·τ|', len=0.28, y=0.16, x=1.02)),
    name='align', showlegend=False,
), row=3, col=1)
if len(peaks_idx):
    fig.add_trace(go.Scatter(
        x=eps_win[peaks_idx], y=pr3_win[peaks_idx], mode='markers',
        marker=dict(symbol='diamond', size=13, color='black', line=dict(width=1, color='white')),
        name='PR₃ peaks',
    ), row=3, col=1)

# Timing vlines on all rows
timing_lines = [
    (fd_end, 'orange', 'fd_end'),
    (R_p101_999['timing']['sd_onset'], 'teal', 'sd_onset'),
    (grok_p, 'black', 'grok'),
    (manual_wiggle, 'red', 'manual wiggle'),
    (manual_terminal, 'purple', 'manual terminal'),
]
for ep, color, _ in timing_lines:
    if ep is not None:
        for r in (1, 2, 3):
            fig.add_vline(x=ep, line=dict(color=color, width=1, dash='dot'), row=r, col=1)

fig.update_xaxes(title_text='epoch', row=3, col=1)
fig.update_yaxes(title_text='s (normalized)', row=1, col=1)
fig.update_yaxes(title_text='ds/dt', row=2, col=1)
fig.update_yaxes(title_text='PR₃', range=[0.95, 3.05], row=3, col=1)
fig.update_layout(
    height=820, width=1100,
    title_text=f'p101/s999/ds598 — manual vs algorithmic cross-check<br>'
               '<sup>orange=fd_end · teal=sd_onset · black=grok · red=manual_wiggle · purple=manual_terminal</sup>',
)
fig.show()


In [ ]:
# Updated cross-variant scatter — PR₃ peak height vs arc-alignment, now including p101/s999/ds598
RUNS_ALL = RUNS_EXT + [R_p101_999]
variant_class_ext = dict(variant_class)
variant_class_ext['p101/s999/ds598'] = 'long bumpy descent'
colors_all = dict(colors_ext)
colors_all['p101/s999/ds598'] = '#ff7f0e'

fig_s2 = go.Figure()
for R in RUNS_ALL:
    d = DECOMP.get(R['label'])
    if d is None:
        d = compute_arc_decomposition(R['mlp_proj'], R['pt_eps'], smooth_k=20, window=10)
        DECOMP[R['label']] = d
    search_lo = R['timing']['fd_end']
    search_hi = commitment_epoch(R)
    in_win = (R['pt_eps'] >= search_lo) & (R['pt_eps'] <= search_hi)
    pr3_win = R['pr3'][in_win]
    align_win = d['alignment'][in_win]
    peaks_idx, _ = find_peaks(pr3_win, prominence=0.05)
    if len(peaks_idx) == 0:
        continue
    fig_s2.add_trace(go.Scatter(
        x=align_win[peaks_idx], y=pr3_win[peaks_idx], mode='markers',
        marker=dict(size=13, color=colors_all[R['label']], line=dict(width=1, color='white')),
        name=f'{R["label"]} ({variant_class_ext[R["label"]]})',
        hovertemplate=f'{R["label"]}<br>align %{{x:.3f}}<br>PR₃ %{{y:.3f}}<extra></extra>',
    ))

fig_s2.add_vline(x=0.4, line=dict(color='grey', width=1, dash='dot'))
fig_s2.add_vline(x=0.7, line=dict(color='grey', width=1, dash='dot'))
fig_s2.add_annotation(x=0.2, y=3.0, text='off-arc<br>(wobble)', showarrow=False, font=dict(color='steelblue'))
fig_s2.add_annotation(x=0.85, y=3.0, text='on-arc<br>(curvature)', showarrow=False, font=dict(color='firebrick'))

fig_s2.update_layout(
    title='PR₃ peaks — height vs arc-alignment (5 variants including p101/s999/ds598)',
    xaxis=dict(title='alignment  |u₁ · τ|', range=[0, 1.02]),
    yaxis=dict(title='PR₃ at peak', range=[0.95, 3.05]),
    height=500, width=820,
)
fig_s2.show()

# Summary table across all five variants
print()
print('On-arc peak count in [fd_end, commit]:')
print(f'{"variant":22s}  {"class":22s}  {"on-arc":>7s}  {"mixed":>6s}  {"off-arc":>8s}')
print('-' * 75)
for R in RUNS_ALL:
    d = DECOMP[R['label']]
    search_lo = R['timing']['fd_end']
    search_hi = commitment_epoch(R)
    in_win = (R['pt_eps'] >= search_lo) & (R['pt_eps'] <= search_hi)
    pr3_win = R['pr3'][in_win]
    align_win = d['alignment'][in_win]
    peaks_idx, _ = find_peaks(pr3_win, prominence=0.05)
    on_arc = sum(1 for pk in peaks_idx if align_win[pk] > 0.7)
    mixed  = sum(1 for pk in peaks_idx if 0.4 <= align_win[pk] <= 0.7)
    off_arc = sum(1 for pk in peaks_idx if align_win[pk] < 0.4)
    print(f'{R["label"]:22s}  {variant_class_ext[R["label"]]:22s}  {on_arc:>7d}  {mixed:>6d}  {off_arc:>8d}')


---

## 14. Valley-depth topology — what sits between consecutive PR₃ peaks

**User observation (2026-04-18).** The MLP-trajectory PR₃ traces across variants show three distinct inter-peak topologies:

- **Canon / fast-clean.** Deep valley between on-arc peaks — PR₃ drops clearly before rising. Clean straightening → commit.
- **p101/s999/ds598 (long bumpy descent).** Valley is shallow — a notch, not a trough. Trajectory partially straightens, resumes curving. Delayed eventual commit.
- **p59/s485/ds598 (supercritical).** "Low peak between two high peaks." The valley-that-wasn't bulged into a third peak. Reads as an aborted commit: trajectory tried to straighten, failed, curled back.

**Diagnostic.** For each adjacent pair of PR₃ peaks (prominence ≥ 0.05) in `[fd_end, commit]`:

- `flank_pr3` — min of the two flanking peak heights
- `valley_pr3` — min PR₃ in the segment between peaks
- `depth = flank_pr3 − valley_pr3` — absolute drop between peaks
- `reach_to_one = valley_pr3 − 1.0` — how close the valley comes to full straightening (0 = fully straight, higher = less straight)
- **sub-threshold intervening maxes** — local maxes with prominence ≥ 0.02 strictly between the two peaks. A non-empty set is the *aborted-commit* signature (a failed straightening attempt that didn't rise to prominence 0.05).

Classification per inter-peak segment:
- `clean_valley` — no intervening max
- `aborted_commit` — exactly one intervening max
- `irregular` — multiple intervening maxes

Two patterns to watch for:
1. **Prominent-peak alternation** (p59-style): when `find_peaks` at 0.05 already returns alternating high/low peaks, the low peak *is* a failed-straightening attempt promoted to its own peak. The flanking high→low and low→high transitions should both show shallow depth.
2. **Sub-threshold intervening max**: a bump too small for 0.05 but visible at 0.02 — a stalled straightening that never resolved.

The question: does `reach_to_one` sort variants by health, and does the p59 "low middle peak" manifest as shallow-depth paired transitions?

In [ ]:
def analyze_valley_topology(pr3, eps, peaks_idx, low_prom=0.02):
    """For each adjacent pair of peaks, characterize the inter-peak topology.

    peaks_idx : indices into pr3/eps (from find_peaks at higher prominence)
    low_prom  : prominence threshold for detecting sub-threshold intervening maxes
    Returns list of dicts, one per adjacent-peak pair.
    """
    transitions = []
    for i in range(len(peaks_idx) - 1):
        lo, hi = int(peaks_idx[i]), int(peaks_idx[i + 1])
        if hi - lo < 3:
            continue
        seg = pr3[lo:hi + 1]
        interior = seg[1:-1]
        mid_max_rel, _ = find_peaks(interior, prominence=low_prom)
        mid_max_idx = mid_max_rel + 1 + lo
        valley_rel = int(np.argmin(seg))
        valley_idx = lo + valley_rel
        flank = float(min(pr3[lo], pr3[hi]))
        depth = float(flank - pr3[valley_idx])
        reach = float(pr3[valley_idx] - 1.0)
        if len(mid_max_idx) == 0:
            kind = 'clean_valley'
        elif len(mid_max_idx) == 1:
            kind = 'aborted_commit'
        else:
            kind = 'irregular'
        transitions.append({
            'peak_a_ep': int(eps[lo]), 'peak_b_ep': int(eps[hi]),
            'peak_a_pr3': float(pr3[lo]), 'peak_b_pr3': float(pr3[hi]),
            'valley_ep': int(eps[valley_idx]), 'valley_pr3': float(pr3[valley_idx]),
            'flank_pr3': flank, 'depth': depth, 'reach_to_one': reach,
            'intervening_eps': [int(eps[m]) for m in mid_max_idx],
            'intervening_pr3': [float(pr3[m]) for m in mid_max_idx],
            'kind': kind,
        })
    return transitions


TOPOLOGY = {}
print(f'{"variant":20s}  {"cls":22s}  peak heights (on/off)')
print('-' * 90)
for R in RUNS_ALL:
    search_lo = R['timing']['fd_end']
    search_hi = commitment_epoch(R)
    in_win = (R['pt_eps'] >= search_lo) & (R['pt_eps'] <= search_hi)
    pr3_win = R['pr3'][in_win]
    eps_win = R['pt_eps'][in_win]
    align_win = DECOMP[R['label']]['alignment'][in_win]
    peaks_idx, _ = find_peaks(pr3_win, prominence=0.05)
    cls = variant_class_ext[R['label']]
    seq = []
    for pk in peaks_idx:
        a = align_win[pk]
        tag = 'on' if a > 0.7 else ('off' if a < 0.4 else 'mix')
        seq.append(f'{pr3_win[pk]:.2f}({tag}@{int(eps_win[pk])})')
    print(f'{R["label"]:20s}  {cls:22s}  {"  →  ".join(seq) if seq else "(none)"}')

print()
print(f'{"variant":20s}  {"pair (ep → ep)":24s}  {"flank":>6s}  {"valley":>6s}  {"depth":>6s}  {"r→1":>6s}  kind')
print('-' * 100)
for R in RUNS_ALL:
    search_lo = R['timing']['fd_end']
    search_hi = commitment_epoch(R)
    in_win = (R['pt_eps'] >= search_lo) & (R['pt_eps'] <= search_hi)
    pr3_win = R['pr3'][in_win]
    eps_win = R['pt_eps'][in_win]
    peaks_idx, _ = find_peaks(pr3_win, prominence=0.05)
    trans = analyze_valley_topology(pr3_win, eps_win, peaks_idx, low_prom=0.02)
    TOPOLOGY[R['label']] = trans
    if len(trans) == 0:
        print(f'{R["label"]:20s}  (no adjacent peak pairs; n_peaks={len(peaks_idx)})')
    for t in trans:
        pair = f'{t["peak_a_ep"]} → {t["peak_b_ep"]}'
        intv = '' if t['kind'] == 'clean_valley' else f'  intv={t["intervening_eps"]}'
        print(f'{R["label"]:20s}  {pair:24s}  {t["flank_pr3"]:>6.3f}  {t["valley_pr3"]:>6.3f}  {t["depth"]:>6.3f}  {t["reach_to_one"]:>6.3f}  {t["kind"]}{intv}')


In [ ]:
# Per-variant PR₃ with peak (on/off/mixed) + valley (clean/aborted/irregular) + intervening-max overlays
fig = make_subplots(
    rows=3, cols=2,
    subplot_titles=[f'{R["label"]}  ({variant_class_ext[R["label"]]})' for R in RUNS_ALL] + [''] * max(0, 6 - len(RUNS_ALL)),
    shared_yaxes=False, vertical_spacing=0.10, horizontal_spacing=0.09,
)

valley_color = {'clean_valley': 'seagreen', 'aborted_commit': 'darkorange', 'irregular': 'purple'}

for i, R in enumerate(RUNS_ALL):
    r = i // 2 + 1
    c = i % 2 + 1
    eps = R['pt_eps']
    pr3 = R['pr3']
    d = DECOMP[R['label']]
    align = d['alignment']
    commit = commitment_epoch(R)
    search_lo = R['timing']['fd_end']
    in_win = (eps >= search_lo) & (eps <= commit)
    pr3_win = pr3[in_win]
    eps_win = eps[in_win]
    align_win = align[in_win]

    fig.add_trace(go.Scatter(
        x=eps, y=pr3, mode='lines', line=dict(color='rgba(120,120,120,0.5)', width=1.2),
        name='PR₃', showlegend=(i == 0), legendgroup='pr3',
    ), row=r, col=c)

    peaks_idx, _ = find_peaks(pr3_win, prominence=0.05)
    if len(peaks_idx):
        pk_align = align_win[peaks_idx]
        pk_colors = ['firebrick' if a > 0.7 else ('steelblue' if a < 0.4 else 'grey') for a in pk_align]
        fig.add_trace(go.Scatter(
            x=eps_win[peaks_idx], y=pr3_win[peaks_idx], mode='markers',
            marker=dict(symbol='diamond', size=13, color=pk_colors, line=dict(width=1, color='white')),
            name='peak (on=red / off=blue / mix=grey)', showlegend=(i == 0), legendgroup='peak',
            hovertemplate='ep %{x}<br>PR₃ %{y:.3f}<extra></extra>',
        ), row=r, col=c)

    for t in TOPOLOGY[R['label']]:
        fig.add_trace(go.Scatter(
            x=[t['valley_ep']], y=[t['valley_pr3']], mode='markers',
            marker=dict(symbol='circle', size=11, color=valley_color[t['kind']], line=dict(width=1, color='white')),
            name=f'valley: {t["kind"]}', showlegend=False, legendgroup=f'v_{t["kind"]}',
            hovertemplate=f'{t["kind"]}<br>ep %{{x}}<br>PR₃ %{{y:.3f}}<br>depth {t["depth"]:.3f}<extra></extra>',
        ), row=r, col=c)
        for (iep, ipr3) in zip(t['intervening_eps'], t['intervening_pr3']):
            fig.add_trace(go.Scatter(
                x=[iep], y=[ipr3], mode='markers',
                marker=dict(symbol='triangle-up', size=12, color='darkorange', line=dict(width=1, color='white')),
                name='intervening max', showlegend=False, legendgroup='intv',
                hovertemplate='intervening max<br>ep %{x}<br>PR₃ %{y:.3f}<extra></extra>',
            ), row=r, col=c)

    fig.add_vline(x=commit, line=dict(color='green', width=1, dash='dot'), row=r, col=c)
    fig.update_xaxes(title_text='epoch', row=r, col=c)
    fig.update_yaxes(title_text='PR₃', range=[0.95, 3.05], row=r, col=c)

if len(RUNS_ALL) < 6:
    fig.update_xaxes(visible=False, row=3, col=2)
    fig.update_yaxes(visible=False, row=3, col=2)

fig.update_layout(
    height=950, width=1200,
    title_text='Inter-peak valley topology — valleys colored by kind (green=clean, orange=aborted, purple=irregular); △ marks sub-threshold intervening maxes',
)
fig.show()


In [ ]:
# Summary scatter — one point per inter-peak transition; depth vs reach-to-one; marker shape by kind
kind_symbol = {'clean_valley': 'circle', 'aborted_commit': 'triangle-up', 'irregular': 'x'}

fig_v = go.Figure()
for R in RUNS_ALL:
    trans = TOPOLOGY[R['label']]
    if len(trans) == 0:
        continue
    xs = [t['reach_to_one'] for t in trans]
    ys = [t['depth'] for t in trans]
    syms = [kind_symbol[t['kind']] for t in trans]
    hov = [f'{R["label"]}<br>{t["peak_a_ep"]} → {t["peak_b_ep"]}<br>{t["kind"]}<br>valley PR₃ {t["valley_pr3"]:.3f}<br>depth {t["depth"]:.3f}' for t in trans]
    fig_v.add_trace(go.Scatter(
        x=xs, y=ys, mode='markers',
        marker=dict(color=colors_all[R['label']], symbol=syms, size=14, line=dict(width=1, color='white')),
        name=f'{R["label"]} ({variant_class_ext[R["label"]]})',
        hovertext=hov, hoverinfo='text',
    ))

fig_v.add_annotation(x=0.15, y=1.6, text='deep valley,<br>low reach<br>(clean commit)', showarrow=False, font=dict(color='seagreen'))
fig_v.add_annotation(x=1.6, y=0.15, text='shallow notch,<br>high reach<br>(bumpy/aborted)', showarrow=False, font=dict(color='darkorange'))
fig_v.update_layout(
    title='Inter-peak valley depth vs reach-to-one  (shape: ○ clean  △ aborted  × irregular)',
    xaxis=dict(title='reach_to_one  (valley PR₃ − 1.0)', range=[-0.05, 2.1]),
    yaxis=dict(title='depth  (flank PR₃ − valley PR₃)', range=[-0.05, 2.1]),
    height=520, width=860,
)
fig_v.show()


In [ ]:
# Per-variant stacked bar — count of transitions by kind
kinds = ['clean_valley', 'aborted_commit', 'irregular']
kind_color = {'clean_valley': 'seagreen', 'aborted_commit': 'darkorange', 'irregular': 'purple'}

labels = [R['label'] for R in RUNS_ALL]
class_labels = [variant_class_ext[R['label']] for R in RUNS_ALL]

fig_b = go.Figure()
for k in kinds:
    counts = [sum(1 for t in TOPOLOGY[R['label']] if t['kind'] == k) for R in RUNS_ALL]
    fig_b.add_trace(go.Bar(
        x=[f'{lab}<br>({cls})' for lab, cls in zip(labels, class_labels)],
        y=counts, name=k, marker=dict(color=kind_color[k]),
    ))

fig_b.update_layout(
    title='Inter-peak transition kinds by variant',
    barmode='stack', yaxis=dict(title='count'),
    height=460, width=960,
)
fig_b.show()


---

## 15. 3D ribbon view — raw trajectory with smoothed backbone

**Goal.** Render the "global slow-moving line with component orbits around it" intuition directly. §12 defined the math (smoothed tangent τ and local-window PC1 u₁); here we show the geometry.

- **Raw** (thin, grey) — full MLP parameter trajectory projected onto PC1–PC3.
- **Backbone** (thick, epoch-colored) — same trajectory with a ±20-checkpoint moving average applied. This is the slow arc.
- **Residual** = raw − backbone. Plotted below as `||residual(t)||` in full 10-D — spikes = local off-arc excursions; near-zero = clean tracking of the backbone.

Peaks from §12/14 are overlaid on the backbone (♦ on=red / off=blue / mixed=grey) and on the residual trace so you can check: *do the PR₃ peaks sit where the raw line leaves the backbone?*

Fast-rigid prediction (p109): backbone nearly straight, residual small throughout.
Classical-clean (canon): one or two strong backbone curves at commit events, residual lining up with on-arc peaks.
Overshooter (p59): backbone straight early, then the backbone *itself* meanders — residual stays elevated through the wobbling plateau.

In [ ]:
def compute_position_backbone(proj, smooth_k=20):
    """Moving-average smoothing of a (n, k) trajectory. Returns (n, k) backbone."""
    proj = np.asarray(proj, dtype=np.float64)
    n, k = proj.shape
    smoothed = np.zeros_like(proj)
    for t in range(n):
        lo = max(0, t - smooth_k)
        hi = min(n, t + smooth_k + 1)
        smoothed[t] = proj[lo:hi].mean(axis=0)
    return smoothed


BACKBONE = {}
print(f'{"variant":20s}  n_epochs  residual_mag: mean / max (@epoch)')
print('-' * 78)
for R in RUNS_ALL:
    proj = R['mlp_proj']  # (n_epochs, 10)
    backbone = compute_position_backbone(proj, smooth_k=20)
    residual = proj - backbone
    residual_mag = np.linalg.norm(residual, axis=1)
    BACKBONE[R['label']] = dict(backbone=backbone, residual=residual, residual_mag=residual_mag)
    peak_ep = int(R['pt_eps'][int(np.argmax(residual_mag))])
    print(f'{R["label"]:20s}  {len(proj):>7d}  {residual_mag.mean():.4f} / {residual_mag.max():.4f} (@ep {peak_ep})')


In [ ]:
# 3D ribbon: raw (grey) + backbone (epoch-colored) + peak & commit markers
fig = make_subplots(
    rows=3, cols=2,
    specs=[[{'type': 'scene'}, {'type': 'scene'}],
           [{'type': 'scene'}, {'type': 'scene'}],
           [{'type': 'scene'}, None]],
    subplot_titles=[f'{R["label"]}  ({variant_class_ext[R["label"]]})' for R in RUNS_ALL],
    horizontal_spacing=0.04, vertical_spacing=0.06,
)

for i, R in enumerate(RUNS_ALL):
    r = i // 2 + 1
    c = i % 2 + 1
    eps = R['pt_eps']
    proj3 = R['mlp_proj'][:, :3]
    backbone3 = BACKBONE[R['label']]['backbone'][:, :3]
    commit = commitment_epoch(R)
    commit_idx = int(np.argmin(np.abs(eps - commit)))

    fig.add_trace(go.Scatter3d(
        x=proj3[:, 0], y=proj3[:, 1], z=proj3[:, 2],
        mode='lines', line=dict(color='rgba(50,50,50,0.80)', width=3),
        name='raw', showlegend=(i == 0), legendgroup='raw',
        hovertext=[f'ep {int(e)}' for e in eps], hoverinfo='text',
    ), row=r, col=c)

    line_cfg = dict(color=eps, colorscale='Viridis', width=6, showscale=(i == 0))
    if i == 0:
        line_cfg['colorbar'] = dict(title='epoch', x=1.02, len=0.6, y=0.5)
    fig.add_trace(go.Scatter3d(
        x=backbone3[:, 0], y=backbone3[:, 1], z=backbone3[:, 2],
        mode='lines', line=line_cfg,
        name='backbone', showlegend=(i == 0), legendgroup='backbone',
        hovertext=[f'ep {int(e)} (backbone)' for e in eps], hoverinfo='text',
    ), row=r, col=c)

    fig.add_trace(go.Scatter3d(
        x=[backbone3[commit_idx, 0]], y=[backbone3[commit_idx, 1]], z=[backbone3[commit_idx, 2]],
        mode='markers', marker=dict(size=7, color='green', symbol='diamond'),
        name='commit', showlegend=(i == 0), legendgroup='commit',
        hovertext=f'commit ep {int(commit)}', hoverinfo='text',
    ), row=r, col=c)

    # On-arc / off-arc / mixed PR₃ peaks on the backbone
    search_lo = R['timing']['fd_end']
    in_win = (eps >= search_lo) & (eps <= commit)
    pr3_win = R['pr3'][in_win]
    eps_win = eps[in_win]
    align_win = DECOMP[R['label']]['alignment'][in_win]
    peaks_idx, _ = find_peaks(pr3_win, prominence=0.05)
    for pk in peaks_idx:
        a = align_win[pk]
        color = 'firebrick' if a > 0.7 else ('steelblue' if a < 0.4 else 'grey')
        full_idx = int(np.argmin(np.abs(eps - eps_win[pk])))
        fig.add_trace(go.Scatter3d(
            x=[backbone3[full_idx, 0]], y=[backbone3[full_idx, 1]], z=[backbone3[full_idx, 2]],
            mode='markers', marker=dict(size=6, color=color, symbol='diamond'),
            showlegend=False,
            hovertext=f'peak ep {int(eps_win[pk])}  PR₃ {pr3_win[pk]:.3f}  align {a:.3f}',
            hoverinfo='text',
        ), row=r, col=c)

for row in (1, 2, 3):
    for col in (1, 2):
        if row == 3 and col == 2:
            continue
        fig.update_scenes(aspectmode='data',
                          xaxis_title='PC1', yaxis_title='PC2', zaxis_title='PC3',
                          row=row, col=col)

fig.update_layout(
    height=1300, width=1200,
    title_text='MLP trajectory — raw (grey) + smoothed backbone (epoch-colored) + ♦ peaks (on=red / off=blue / mix=grey) + ♦ commit (green)',
)
fig.show()


In [ ]:
# Companion: wobble magnitude over epoch — full 10-D residual
fig_w = make_subplots(
    rows=3, cols=2,
    subplot_titles=[f'{R["label"]}  ({variant_class_ext[R["label"]]})' for R in RUNS_ALL] + [''] * max(0, 6 - len(RUNS_ALL)),
    shared_yaxes=False, vertical_spacing=0.10, horizontal_spacing=0.08,
)

for i, R in enumerate(RUNS_ALL):
    r = i // 2 + 1
    c = i % 2 + 1
    eps = R['pt_eps']
    res_mag = BACKBONE[R['label']]['residual_mag']
    commit = commitment_epoch(R)

    fig_w.add_trace(go.Scatter(
        x=eps, y=res_mag, mode='lines', line=dict(color='#d62728', width=1.5),
        name='||raw − backbone||', showlegend=(i == 0),
        hovertemplate='ep %{x}<br>wobble %{y:.4f}<extra></extra>',
    ), row=r, col=c)

    search_lo = R['timing']['fd_end']
    in_win = (eps >= search_lo) & (eps <= commit)
    pr3_win = R['pr3'][in_win]
    eps_win = eps[in_win]
    align_win = DECOMP[R['label']]['alignment'][in_win]
    peaks_idx, _ = find_peaks(pr3_win, prominence=0.05)
    for pk in peaks_idx:
        a = align_win[pk]
        color = 'firebrick' if a > 0.7 else ('steelblue' if a < 0.4 else 'grey')
        full_idx = int(np.argmin(np.abs(eps - eps_win[pk])))
        fig_w.add_trace(go.Scatter(
            x=[eps_win[pk]], y=[res_mag[full_idx]], mode='markers',
            marker=dict(symbol='diamond', size=11, color=color, line=dict(width=1, color='white')),
            showlegend=False,
            hovertemplate=f'peak ep {int(eps_win[pk])}<br>PR₃ {pr3_win[pk]:.3f}<br>align {a:.3f}<extra></extra>',
        ), row=r, col=c)

    fig_w.add_vline(x=commit, line=dict(color='green', width=1, dash='dot'), row=r, col=c)
    fig_w.update_xaxes(title_text='epoch', row=r, col=c)
    fig_w.update_yaxes(title_text='||residual||', row=r, col=c)

if len(RUNS_ALL) < 6:
    fig_w.update_xaxes(visible=False, row=3, col=2)
    fig_w.update_yaxes(visible=False, row=3, col=2)

fig_w.update_layout(
    height=900, width=1200,
    title_text='Wobble magnitude — ||raw − backbone|| in full 10-D PC space',
)
fig_w.show()


---

## 16. Cross-site Class Centroid PR₃ — the handoff diagnostic

**Goal.** §12–15 lived on Panel 1 (trajectory rolling PR₃ — the shape of the optimization path). The hypothesised Attn/MLP handoff lives on **Panel 2** (Class Centroid PR₃ — the geometry of per-class activation centroids at each site). This section pulls Panel-2 data directly and tests:

**Handoff hypothesis.** During second descent, Attention sheds activation-geometry dimensionality (Attn PR₃: ~3 → ~2) *while* MLP and ResidPost expand (MLP PR₃: ~2 → ~2.8). Representation load transfers; MLPs use the extra space to reorganise. Expect anti-correlated Attn↓ / MLP↑ in a bounded window near commitment.

**Sites.** `attn_out`, `mlp_out`, `resid_post` from the repr_geometry summary (`{site}_pca_var_pc1/pc2/pc3` → PR₃ via (Σfᵢ)²/Σfᵢ²). `resid_pre` is the pre-attention residual — zero centroid spread here (1D embedding), not plotted.

**Handoff metric.** Scalar flux `H(t) = -d(Attn_PR₃)/dt · d(MLP_PR₃)/dt`. Positive = anti-correlated motion = handoff. Magnitude reflects the intensity of the exchange. Rolling-smoothed to suppress per-step noise.

**Spark hypothesis (overlay).** Freq-group saddle-entry events — epoch where a group's `Win_pr3` first drops below 2.7 (disc → saddle) — are candidates for what *triggers* the handoff. Do group entries precede handoff onset? Per-variant predictions:

- **p113/canon** — clear anti-correlated Attn↓/MLP↑ ~10k; group entries cluster before.
- **p109/fast-rigid** — minimal handoff amplitude (already compact); groups enter nearly simultaneously.
- **p101/s485/ds999** — mild handoff at ~7k; one freestanding group (freq 27) may entry-lag.
- **p101/s999/ds598 (bumpy)** — possibly *two* handoff events during the plateau.
- **p59** — **no anti-correlation**. Expect H ≈ 0 or negative. MLP and ResidPost rise together — compensation, not handoff.

Per-head attention deferred — centroid lives on `attn_out` (residual read, population signal). See `feedback_dim_dynamics_panels.md`.

In [ ]:
from miscope.visualization.renderers.dimensionality_dynamics import _compute_pr3

_CENTROID_SITES = ['attn_out', 'mlp_out', 'resid_post']
_SADDLE_THRESHOLD = 2.7  # Win_pr3 crossing this downward = saddle entry


def load_centroid_pr3(variant):
    """Per-site Class Centroid PR₃ series from repr_geometry summary."""
    rg = variant.artifacts.load_summary('repr_geometry')
    out = {'epochs': np.asarray(rg['epochs'], dtype=int)}
    for site in _CENTROID_SITES:
        f1 = np.asarray(rg[f'{site}_pca_var_pc1'])
        f2 = np.asarray(rg[f'{site}_pca_var_pc2'])
        f3 = np.asarray(rg[f'{site}_pca_var_pc3'])
        out[site] = _compute_pr3(f1, f2, f3)
    return out


def detect_saddle_entries(variant, threshold=_SADDLE_THRESHOLD):
    """First epoch each freq group's Win_pr3 crosses below threshold.
    Returns list of (group_freq, group_size, entry_epoch) — missing groups: never entered.
    """
    loader = ArtifactLoader(str(variant.variant_dir / 'artifacts'))
    wg = loader.load_cross_epoch('freq_group_weight_geometry')
    epochs = np.asarray(wg['epochs'], dtype=int)
    freqs = np.asarray(wg['group_freqs']).ravel()
    sizes = np.asarray(wg['group_sizes']).ravel()
    win_pr3 = np.asarray(wg['Win_pr3'])  # (n_epochs, n_groups)
    entries = []
    for g_idx, (f, sz) in enumerate(zip(freqs, sizes)):
        below = win_pr3[:, g_idx] < threshold
        if not below.any():
            continue
        first = int(np.argmax(below))
        entries.append({'freq': int(f), 'size': int(sz), 'epoch': int(epochs[first])})
    return entries


def rolling_smooth(x, k=5):
    """Centered moving average. Returns same length."""
    x = np.asarray(x, dtype=np.float64)
    n = len(x)
    out = np.zeros(n)
    for t in range(n):
        lo, hi = max(0, t - k), min(n, t + k + 1)
        out[t] = x[lo:hi].mean()
    return out


def handoff_flux(pr3_attn, pr3_mlp, eps, smooth_k=5):
    """H(t) = -d(Attn)/dt · d(MLP)/dt after smoothing. Positive = anti-correlated (handoff)."""
    eps = np.asarray(eps, dtype=np.float64)
    a = rolling_smooth(pr3_attn, smooth_k)
    m = rolling_smooth(pr3_mlp, smooth_k)
    da = np.gradient(a, eps)
    dm = np.gradient(m, eps)
    return -da * dm, da, dm


CENTROID_PR3 = {}
SADDLE_ENTRIES = {}
HANDOFF = {}
print(f'{"variant":20s}  n_epochs  H_max (@ep)        groups_entered / total')
print('-' * 78)
for R in RUNS_ALL:
    cpr3 = load_centroid_pr3(R['variant'])
    entries = detect_saddle_entries(R['variant'])
    H, da, dm = handoff_flux(cpr3['attn_out'], cpr3['mlp_out'], cpr3['epochs'], smooth_k=5)
    CENTROID_PR3[R['label']] = cpr3
    SADDLE_ENTRIES[R['label']] = entries
    HANDOFF[R['label']] = dict(H=H, da_attn=da, dm_mlp=dm, eps=cpr3['epochs'])
    loader_wg = ArtifactLoader(str(R['variant'].variant_dir / 'artifacts'))
    wg_ = loader_wg.load_cross_epoch('freq_group_weight_geometry')
    n_groups = len(wg_['group_freqs'])
    h_peak_idx = int(np.argmax(H))
    print(f'{R["label"]:20s}  {len(cpr3["epochs"]):>7d}  '
          f'{H[h_peak_idx]:+.2e} (@ep {int(cpr3["epochs"][h_peak_idx])})  '
          f'{len(entries):>3d} / {n_groups}')


In [ ]:
# 5-variant panel: per row, 2 cols — (PR₃ traces + saddle-entry markers)  (H flux + commitment)
fig = make_subplots(
    rows=len(RUNS_ALL), cols=2,
    subplot_titles=sum([[f'{R["label"]}  PR₃ per site  ({variant_class_ext[R["label"]]})',
                         f'{R["label"]}  handoff flux H(t)'] for R in RUNS_ALL], []),
    horizontal_spacing=0.08, vertical_spacing=0.06,
    shared_xaxes=False,
)

site_colors = {'attn_out': '#2ca02c', 'mlp_out': '#d62728', 'resid_post': '#9467bd'}
site_labels = {'attn_out': 'Attn', 'mlp_out': 'MLP', 'resid_post': 'ResidPost'}

for i, R in enumerate(RUNS_ALL):
    label = R['label']
    row = i + 1
    cpr3 = CENTROID_PR3[label]
    eps = cpr3['epochs']
    commit = commitment_epoch(R)
    show_legend = (i == 0)

    # --- Col 1: per-site PR₃ traces ---
    for site in _CENTROID_SITES:
        fig.add_trace(go.Scatter(
            x=eps, y=cpr3[site], mode='lines',
            line=dict(color=site_colors[site], width=2),
            name=site_labels[site], legendgroup=site, showlegend=show_legend,
            hovertemplate=f'{site_labels[site]}<br>ep %{{x}}<br>PR₃ %{{y:.3f}}<extra></extra>',
        ), row=row, col=1)

    # Saddle-entry markers at their entry epochs (y = 2.85 reference band)
    for e in SADDLE_ENTRIES[label]:
        fig.add_trace(go.Scatter(
            x=[e['epoch']], y=[2.9], mode='markers',
            marker=dict(symbol='triangle-down', size=11, color='#1f77b4',
                        line=dict(width=1, color='white')),
            showlegend=False,
            hovertemplate=f'group f={e["freq"]} (n={e["size"]}) enters saddle<br>ep {e["epoch"]}<extra></extra>',
        ), row=row, col=1)

    fig.add_vline(x=commit, line=dict(color='green', width=1, dash='dot'), row=row, col=1)
    fig.add_hline(y=2.0, line=dict(color='rgba(0,0,0,0.18)', dash='dot'), row=row, col=1)
    fig.add_hline(y=3.0, line=dict(color='rgba(0,0,0,0.18)', dash='dot'), row=row, col=1)
    fig.update_yaxes(range=[1.0, 3.2], title_text='PR₃', row=row, col=1)
    fig.update_xaxes(title_text='epoch', row=row, col=1)

    # --- Col 2: handoff flux H(t) ---
    H = HANDOFF[label]['H']
    fig.add_trace(go.Scatter(
        x=eps, y=H, mode='lines', line=dict(color='#ff7f0e', width=1.8),
        name='H = -d(Attn)·d(MLP)', legendgroup='H', showlegend=show_legend,
        hovertemplate='ep %{x}<br>H %{y:+.2e}<extra></extra>',
    ), row=row, col=2)
    fig.add_hline(y=0.0, line=dict(color='black', width=0.8), row=row, col=2)
    fig.add_vline(x=commit, line=dict(color='green', width=1, dash='dot'), row=row, col=2)

    # Saddle-entry markers on H as well — do they precede H spike?
    for e in SADDLE_ENTRIES[label]:
        full_idx = int(np.argmin(np.abs(eps - e['epoch'])))
        fig.add_trace(go.Scatter(
            x=[e['epoch']], y=[H[full_idx]], mode='markers',
            marker=dict(symbol='triangle-down', size=10, color='#1f77b4',
                        line=dict(width=1, color='white')),
            showlegend=False,
            hovertemplate=f'group f={e["freq"]} enters saddle<br>ep {e["epoch"]}<br>H {H[full_idx]:+.2e}<extra></extra>',
        ), row=row, col=2)

    fig.update_yaxes(title_text='H(t)', row=row, col=2)
    fig.update_xaxes(title_text='epoch', row=row, col=2)

fig.update_layout(
    height=300 * len(RUNS_ALL), width=1300,
    title_text='Class Centroid PR₃ handoff — Attn (green) / MLP (red) / ResidPost (purple) + ▽ group saddle entries + ♦ commit (green dotted)',
    hovermode='closest',
)
fig.show()


In [ ]:
# Summary table — handoff window characterisation per variant
# Remedies applied:
#   1) H_peak search restricted to [fd_end, commit]; added ∫ max(0, H) dt over the same window.
#   2) Attn-drop: added Δ (smoothed PR₃ change) and duration and their product (amplitude × duration).
print(f'{"variant":20s}  {"class":18s}  {"H_peak":>10s}  {"ep_peak":>7s}  {"∫H+ dt":>10s}  {"grps_before":>11s}  {"attn_drop_window":>20s}  {"Δattn":>7s}  {"dur":>6s}  {"Δ·dur":>9s}')
print('-' * 150)
rows = []
for R in RUNS_ALL:
    label = R['label']
    cpr3 = CENTROID_PR3[label]
    eps = cpr3['epochs']
    H = HANDOFF[label]['H']
    da = HANDOFF[label]['da_attn']
    commit = commitment_epoch(R)

    # --- Windowed H analysis: restrict to [fd_end, commit] (or [fd_end, end] if no commit) ---
    search_lo = R['timing']['fd_end']
    search_hi = commit if commit else int(eps[-1])
    in_win = (eps >= search_lo) & (eps <= search_hi)
    H_win = np.where(in_win, H, -np.inf)
    peak_idx = int(np.argmax(H_win))
    H_peak = float(H[peak_idx])
    ep_peak = int(eps[peak_idx])

    # ∫ max(0, H) dt over the same window — total "handoff activity"
    eps_in = eps[in_win].astype(np.float64)
    H_in = H[in_win]
    H_pos = np.clip(H_in, 0.0, None)
    H_integral = float(np.trapezoid(H_pos, eps_in)) if len(eps_in) > 1 else 0.0

    # --- Attn-drop window: longest contiguous descending run; then amplitude × duration ---
    attn_s = rolling_smooth(cpr3['attn_out'], 5)
    desc = da < -1e-4
    best = (0, 0, 0)  # (len, start_idx, end_idx)
    run_start = None
    for t, b in enumerate(desc):
        if b and run_start is None:
            run_start = t
        elif not b and run_start is not None:
            L = t - run_start
            if L > best[0]:
                best = (L, run_start, t - 1)
            run_start = None
    if run_start is not None:
        L = len(desc) - run_start
        if L > best[0]:
            best = (L, run_start, len(desc) - 1)

    if best[0] > 0:
        s_idx, e_idx = best[1], best[2]
        attn_drop_win = f'[{int(eps[s_idx])}, {int(eps[e_idx])}]'
        delta_attn = float(attn_s[e_idx] - attn_s[s_idx])  # negative = descent
        duration = int(eps[e_idx] - eps[s_idx])
        amp_dur = abs(delta_attn) * duration  # magnitude × span
    else:
        attn_drop_win = '—'
        delta_attn = 0.0
        duration = 0
        amp_dur = 0.0

    # Count group entries before the (now windowed) H peak
    entries = SADDLE_ENTRIES[label]
    groups_before = sum(1 for e in entries if e['epoch'] < ep_peak)

    kls = variant_class_ext[label]
    print(f'{label:20s}  {kls:18s}  {H_peak:+.2e}  {ep_peak:>7d}  {H_integral:+.2e}  '
          f'{groups_before:>4d}/{len(entries):<4d}  {attn_drop_win:>20s}  {delta_attn:+.3f}  {duration:>6d}  {amp_dur:>9.2f}')
    rows.append(dict(label=label, kls=kls,
                     H_peak=H_peak, ep_peak=ep_peak,
                     H_integral=H_integral,
                     groups_before=groups_before, n_entries=len(entries),
                     attn_drop_window=attn_drop_win,
                     delta_attn=delta_attn, duration=duration,
                     amp_dur=amp_dur))

print()
print('Columns:')
print('  H_peak      — max H within [fd_end, commit]; first-descent noise no longer dominates')
print('  ∫H+ dt      — integrated positive handoff flux over [fd_end, commit]; total exchange budget')
print('  grps_before — freq-group saddle entries that occurred before windowed H_peak')
print('  Δattn       — smoothed Attn PR₃ change over longest descending run (negative = drop)')
print('  dur         — duration of that descending run (epochs)')
print('  Δ·dur       — amplitude × duration: rate-duration trade-off scorer')
print()
print('Comparisons:')
print('  ∫H+ dt lets us compare *total* handoff activity (fast-deep vs slow-moderate become commensurable)')
print('  Δ·dur  lets us compare the attention-side amplitude × time budget across variants')


---
## 17 Local-sigmoid falsification — is each PR₃ peak its own saddle?

**Hypothesis under test.** Each prominent MLP PR₃ peak in [fd_end, commit] is a distinct saddle crossing, not just a curvature fluctuation.

**Reframe of prior sections.** §3's global sigmoid fit projects onto `v = (W_T − W_w)/‖·‖` — a *chord* spanning the entire transit. On multi-peak variants the chord averages across several saddles, flattening each individual transit signal into drift (p59, §11; p101/s999, §13). If peaks really are saddles, *local* transits between adjacent PR₃ valleys should each show a clean sigmoid.

**Construction.** For every prominent peak in [fd_end, commit]:
- Left bound = previous PR₃ valley (or `fd_end` for the first peak).
- Right bound = next PR₃ valley (or `commit` for the last peak).
- Local transit vector = normalized displacement between the two bounds *in full MLP parameter space* (same space §3 uses, not the 10-D PCA).
- Project the segment trajectory onto that vector; fit sigmoid and a chord-linear baseline.

**Decision rule.**
- `R²_sig` high AND `R²_sig − R²_lin` materially positive → segment is sigmoidal → consistent with a real saddle at the peak.
- `R²_sig ≈ R²_lin` → segment is approximately linear drift → peak is curvature without topology, or the sigmoid sits outside the valley-bounded segment.

**Pair with §12 alignment.** Cross-tabulate local sigmoidality against on-arc / off-arc classification from §12 — on-arc peaks ought to carry the sigmoidal signal; off-arc peaks should look drift-like or off-diagonal.


In [ ]:
# §17 — local-sigmoid helpers. Built around §14 valley topology and §12 alignment.
from scipy.optimize import curve_fit


def _sigmoid(t, A, center, slope, baseline):
    return baseline + A / (1.0 + np.exp(-(t - center) / max(slope, 1e-6)))


def _nearest_idx(epochs_arr, target_ep):
    return int(np.argmin(np.abs(np.asarray(epochs_arr) - target_ep)))


def build_valley_segments(R, topology_for_label, commit_ep):
    """Valley-bounded segments, one per PR₃ peak in [fd_end, commit].

    Peaks re-derived with the same find_peaks(prominence=0.05) as §14 for parity.
    Returns list of dicts keyed for downstream fitting.
    """
    fd_end = R['timing']['fd_end']
    in_win = (R['pt_eps'] >= fd_end) & (R['pt_eps'] <= commit_ep)
    pr3_win = R['pr3'][in_win]
    eps_win = R['pt_eps'][in_win]
    peaks_idx, _ = find_peaks(pr3_win, prominence=0.05)
    peak_eps = [int(eps_win[i]) for i in peaks_idx]
    if not peak_eps:
        return []

    valley_eps = [t['valley_ep'] for t in topology_for_label]
    segments = []
    for i, pk in enumerate(peak_eps):
        ep_lo = fd_end if i == 0 else valley_eps[i - 1]
        ep_hi = commit_ep if i == len(peak_eps) - 1 else valley_eps[i]
        segments.append(dict(segment_idx=i, ep_lo=ep_lo, ep_hi=ep_hi, peak_ep=pk))
    return segments


def fit_local_sigmoid(X, epochs, ep_lo, ep_hi):
    """Local transit projection + sigmoid fit within [ep_lo, ep_hi].

    Transit vector computed in full MLP parameter space. Returns None for segments
    too short for a meaningful fit.
    """
    i_lo = _nearest_idx(epochs, ep_lo)
    i_hi = _nearest_idx(epochs, ep_hi)
    if i_hi - i_lo < 5:
        return None

    delta = X[i_hi] - X[i_lo]
    L = float(np.linalg.norm(delta))
    if L < 1e-12:
        return None
    v_hat = delta / L

    eps_seg = epochs[i_lo:i_hi + 1].astype(np.float64)
    seg = X[i_lo:i_hi + 1]
    s = (seg - X[i_lo]) @ v_hat / L  # endpoints anchored at 0 and 1
    t_norm = (eps_seg - eps_seg[0]) / (eps_seg[-1] - eps_seg[0])

    # Sigmoid fit: amplitude near 1, center in [0,1], slope small = steep
    try:
        popt, _ = curve_fit(
            _sigmoid, t_norm, s,
            p0=[1.0, 0.5, 0.15, 0.0],
            bounds=([0.1, -0.5, 0.01, -0.5], [2.0, 1.5, 1.0, 0.5]),
            maxfev=5000,
        )
        s_sig = _sigmoid(t_norm, *popt)
        ss_res_sig = float(np.sum((s - s_sig) ** 2))
    except Exception:
        popt = None
        s_sig = None
        ss_res_sig = float('inf')

    coef = np.polyfit(t_norm, s, 1)
    s_lin = np.polyval(coef, t_norm)
    ss_res_lin = float(np.sum((s - s_lin) ** 2))

    ss_tot = float(np.sum((s - s.mean()) ** 2)) + 1e-12
    r2_sig = 1.0 - ss_res_sig / ss_tot
    r2_lin = 1.0 - ss_res_lin / ss_tot

    return dict(
        eps=eps_seg, s=s, s_sig=s_sig, s_lin=s_lin,
        L=L, sigmoid_params=popt,
        r2_sig=r2_sig, r2_lin=r2_lin,
        sigmoidality=r2_sig - r2_lin,
    )


In [ ]:
# §17 — run per-variant local sigmoid fits; pair each peak with §12 alignment.
LOCAL_SIGMOID = {}
print(f'{"variant":22s}  {"seg":>3s}  {"window":>24s}  {"peak":>6s}  {"L_local":>9s}  '
      f'{"R²_sig":>7s}  {"R²_lin":>7s}  {"Δ":>7s}  {"align":>6s}  verdict')
print('-' * 120)
for R in RUNS_ALL:
    label = R['label']
    commit = commitment_epoch(R)
    fd_end = R['timing']['fd_end']
    segs = build_valley_segments(R, TOPOLOGY.get(label, []), commit)

    d = DECOMP[label]
    in_win = (R['pt_eps'] >= fd_end) & (R['pt_eps'] <= commit)
    pt_eps_win = R['pt_eps'][in_win]
    align_win = d['alignment'][in_win]

    entries = []
    for seg in segs:
        fit = fit_local_sigmoid(R['X'], R['epochs'], seg['ep_lo'], seg['ep_hi'])
        if fit is None:
            continue

        pk_idx_rel = int(np.argmin(np.abs(pt_eps_win - seg['peak_ep'])))
        peak_align = float(align_win[pk_idx_rel])
        flavor = 'on' if peak_align > 0.7 else ('off' if peak_align < 0.4 else 'mix')

        if fit['sigmoidality'] >= 0.05 and fit['r2_sig'] >= 0.90:
            verdict = 'sigmoidal (saddle-like)'
        elif fit['sigmoidality'] >= 0.02:
            verdict = 'mildly sigmoidal'
        else:
            verdict = 'drift-like (non-saddle)'

        entry = dict(
            segment_idx=seg['segment_idx'], peak_ep=seg['peak_ep'],
            ep_lo=seg['ep_lo'], ep_hi=seg['ep_hi'],
            L=fit['L'], r2_sig=fit['r2_sig'], r2_lin=fit['r2_lin'],
            sigmoidality=fit['sigmoidality'], align=peak_align, flavor=flavor,
            verdict=verdict, fit=fit,
        )
        entries.append(entry)
        pair = f'[{seg["ep_lo"]}, {seg["ep_hi"]}]'
        print(f'{label:22s}  {seg["segment_idx"]:>3d}  {pair:>24s}  {seg["peak_ep"]:>6d}  '
              f'{fit["L"]:>9.3f}  {fit["r2_sig"]:>7.3f}  {fit["r2_lin"]:>7.3f}  '
              f'{fit["sigmoidality"]:>+7.3f}  {peak_align:>6.3f}({flavor})  {verdict}')
    LOCAL_SIGMOID[label] = entries
    if not entries:
        print(f'{label:22s}  (no valley-bounded peak segments)')
    print()


In [ ]:
# §17 — per-variant panel: observed s(t) with sigmoid & linear baselines overlaid,
# one subplot per valley-bounded segment.
n_segs_per = {R['label']: max(1, len(LOCAL_SIGMOID[R['label']])) for R in RUNS_ALL}
max_cols = max(n_segs_per.values())

fig_s17 = make_subplots(
    rows=len(RUNS_ALL), cols=max_cols,
    subplot_titles=sum([
        [(f'{R["label"]}  seg {e["segment_idx"]}  peak@{e["peak_ep"]}  '
          f'Δ={e["sigmoidality"]:+.2f}  ({e["flavor"]})')
         for e in LOCAL_SIGMOID[R['label']]] +
        [''] * (max_cols - len(LOCAL_SIGMOID[R['label']]))
        for R in RUNS_ALL
    ], []),
    vertical_spacing=0.08, horizontal_spacing=0.05,
)

legend_shown = {'s': False, 'sig': False, 'lin': False}
for i, R in enumerate(RUNS_ALL):
    entries = LOCAL_SIGMOID[R['label']]
    for j, entry in enumerate(entries):
        row = i + 1
        col = j + 1
        fit = entry['fit']
        t_norm = (fit['eps'] - fit['eps'][0]) / (fit['eps'][-1] - fit['eps'][0])

        fig_s17.add_trace(go.Scatter(
            x=t_norm, y=fit['s'], mode='lines+markers',
            line=dict(color='#1f77b4', width=1.6), marker=dict(size=4),
            name='observed s_local', legendgroup='s', showlegend=not legend_shown['s'],
            hovertemplate='t %{x:.2f}<br>s %{y:.3f}<extra></extra>',
        ), row=row, col=col)
        legend_shown['s'] = True

        if fit['s_sig'] is not None:
            fig_s17.add_trace(go.Scatter(
                x=t_norm, y=fit['s_sig'], mode='lines',
                line=dict(color='#d62728', width=1.6, dash='dash'),
                name=f'sigmoid fit (R²)', legendgroup='sig', showlegend=not legend_shown['sig'],
                hovertemplate='sigmoid<br>t %{x:.2f}<br>s_fit %{y:.3f}<extra></extra>',
            ), row=row, col=col)
            legend_shown['sig'] = True

        fig_s17.add_trace(go.Scatter(
            x=t_norm, y=fit['s_lin'], mode='lines',
            line=dict(color='grey', width=1.1, dash='dot'),
            name='linear chord (R²)', legendgroup='lin', showlegend=not legend_shown['lin'],
            hovertemplate='linear<br>t %{x:.2f}<br>s_fit %{y:.3f}<extra></extra>',
        ), row=row, col=col)
        legend_shown['lin'] = True

        fig_s17.update_xaxes(title_text='t_norm', row=row, col=col, range=[-0.02, 1.02])
        fig_s17.update_yaxes(title_text='s_local', row=row, col=col)

fig_s17.update_layout(
    height=260 * len(RUNS_ALL), width=320 * max_cols,
    title_text='Local sigmoid fits — one subplot per valley-bounded peak segment',
    hovermode='closest',
)
fig_s17.show()


In [ ]:
# §17 — summary scatter: per-peak local sigmoidality vs §12 arc-alignment.
# Prediction: on-arc peaks (align > 0.7) cluster at high sigmoidality (Δ > 0.05).
fig_s17b = go.Figure()
for R in RUNS_ALL:
    label = R['label']
    col = colors_all.get(label, '#333')
    entries = LOCAL_SIGMOID[label]
    if not entries:
        continue
    fig_s17b.add_trace(go.Scatter(
        x=[e['align'] for e in entries],
        y=[e['sigmoidality'] for e in entries],
        mode='markers+text',
        marker=dict(size=13, color=col, line=dict(width=1, color='white')),
        text=[f'seg{e["segment_idx"]}@{e["peak_ep"]}' for e in entries],
        textposition='top center', textfont=dict(size=9, color='#444'),
        name=f'{label} ({variant_class_ext[label]})',
        hovertemplate=(f'{label}<br>seg %{{text}}<br>align %{{x:.3f}}'
                       f'<br>Δ (R²_sig − R²_lin) %{{y:+.3f}}<extra></extra>'),
    ))

fig_s17b.add_hline(y=0.05, line=dict(color='grey', width=1, dash='dot'),
                   annotation_text='Δ=0.05  (saddle-like threshold)', annotation_position='top right')
fig_s17b.add_vline(x=0.7, line=dict(color='grey', width=1, dash='dot'),
                   annotation_text='align=0.7', annotation_position='top')
fig_s17b.add_vline(x=0.4, line=dict(color='grey', width=1, dash='dot'),
                   annotation_text='align=0.4', annotation_position='bottom')

fig_s17b.update_layout(
    title='Local sigmoidality vs §12 arc-alignment — top-right quadrant = on-arc peak with clean local sigmoid',
    xaxis=dict(title='alignment  |u₁ · τ|  at peak', range=[-0.02, 1.02]),
    yaxis=dict(title='Δ = R²_sig − R²_lin  (local)', zeroline=True),
    height=540, width=880,
)
fig_s17b.show()

# Cross-tabulation
print()
print(f'{"variant":22s}  {"on-arc sig.":>12s}  {"off-arc sig.":>13s}  {"mix sig.":>10s}  (count · mean Δ)')
print('-' * 90)
for R in RUNS_ALL:
    label = R['label']
    entries = LOCAL_SIGMOID[label]
    def _bucket(flv):
        vals = [e['sigmoidality'] for e in entries if e['flavor'] == flv]
        return (len(vals), float(np.mean(vals)) if vals else float('nan'))
    n_on,  m_on  = _bucket('on')
    n_off, m_off = _bucket('off')
    n_mix, m_mix = _bucket('mix')
    def _fmt(n, m):
        return f'{n} · {m:+.3f}' if n else '—'
    print(f'{label:22s}  {_fmt(n_on, m_on):>12s}  {_fmt(n_off, m_off):>13s}  {_fmt(n_mix, m_mix):>10s}')


### 17.1 Expected outcomes

If peaks are genuine saddle crossings:
- **p109 (1 peak, on-arc)** — single segment, high sigmoidality. Matches fast-rigid clean transit.
- **p113 canon (1 on-arc + 1 off-arc)** — on-arc segment sigmoidal; off-arc segment drifts.
- **p101/s485/ds999 (rebounder, 1 on-arc)** — sigmoidal, but overshoots in the terminal region (visible as `s > 1` inside the segment or at its right edge).
- **p101/s999/ds598 (2 on-arc)** — both segments sigmoidal → evidence for a serial-saddle chain.
- **p59 (4 on-arc)** — expected outcome is the sharp test. If all four segments are sigmoidal, this is strong evidence for heteroclinic-chain / tube-segment dynamics. If only the first is sigmoidal and the rest drift, the plateau is not a saddle sequence but sustained non-saddle wobble.

### 17.2 Link to frequency-group centroid trajectories

User observation (2026-04-19): freq-group centroid trajectories echo the global MLP trajectory — each group in its own subspace (different angles), with **"twists" — kinks in otherwise flat 2D paths** at localized epochs.

Under the saddle-as-peak reading, this is the mechanistic substrate: each saddle is a per-group commitment event, and the global PR₃ peak is the aggregate signature of one or more groups crossing simultaneously. Natural next step (§18, not in scope here):

- Fit a local tangent to each freq-group's weight-space centroid trajectory across [peak_ep − w, peak_ep + w].
- Detect kink points per group: where the smoothed tangent direction changes by more than some threshold.
- Check whether global PR₃ peaks coincide with kinks in multiple groups (collective saddle) or single groups (staggered chain).


---
## 17.3 Transit-axis enumerator — velocity, pairwise angles, variant regime

**Revised framing after §17.2 cross-tab.** On-arc vs off-arc does not predict local sigmoidality (within every variant, mean Δ differs by ≤0.005 between flavors). §12 arc-alignment and §17 local sigmoidality measure **orthogonal** aspects of curvature:
- §12: does local direction continue along the global arc?
- §17: does the local segment trace an S-shape along its own chord?

A discrete saddle crossing is exactly a segment that is **off-arc AND sigmoidal** — direction changes *and* the segment follows sigmoid kinematics. So §17.3 enumerates transit axes from **all** valley-bounded segments, not filtered by arc-alignment.

**What we compute.**
1. **Unit transit vector** `v̂_k = (X[ep_hi] − X[ep_lo]) / L` per segment, in full MLP parameter space.
2. **Transit velocity** `L_local / Δepoch` — splits low-Δ cases into straight-fast (p109-like) vs stuck (p59-like).
3. **Pairwise angles** between a variant's local axes (degrees, sign-flip absorbed by `|cos|`) — distinguishes a variant whose segments trace **one** global transit direction (near-parallel axes) from one that traces **several** (well-separated axes = serial saddles).
4. **Variant regime classification** from (mean Δ, mean velocity, n_segs, mean pairwise angle).


In [ ]:
# §17.3 — extract per-segment unit transit vector and pairwise angles.
LOCAL_AXES = {}

def extract_local_axes(R):
    axes = []
    epochs, X = R['epochs'], R['X']
    for entry in LOCAL_SIGMOID[R['label']]:
        i_lo = _nearest_idx(epochs, entry['ep_lo'])
        i_hi = _nearest_idx(epochs, entry['ep_hi'])
        delta = X[i_hi] - X[i_lo]
        L = float(np.linalg.norm(delta))
        if L < 1e-12:
            continue
        v_hat = delta / L
        dur = float(entry['ep_hi'] - entry['ep_lo'])
        velocity = L / dur if dur > 0 else 0.0
        axes.append(dict(
            segment_idx=entry['segment_idx'], peak_ep=entry['peak_ep'],
            ep_lo=entry['ep_lo'], ep_hi=entry['ep_hi'],
            v_hat=v_hat, L=L, duration=dur, velocity=velocity,
            sigmoidality=entry['sigmoidality'],
            r2_sig=entry['r2_sig'], r2_lin=entry['r2_lin'],
            align=entry['align'], flavor=entry['flavor'],
        ))
    return axes


def pairwise_angle_matrix(axes):
    n = len(axes)
    M = np.full((n, n), np.nan)
    for i in range(n):
        for j in range(n):
            c = float(np.clip(np.dot(axes[i]['v_hat'], axes[j]['v_hat']), -1.0, 1.0))
            M[i, j] = float(np.degrees(np.arccos(abs(c))))  # |·| → sign-flip invariant
    return M


print(f'{"variant":22s}  {"seg":>3s}  {"peak":>6s}  {"L":>8s}  {"dur":>6s}  '
      f'{"velocity":>10s}  {"Δ":>7s}  {"flavor":>6s}')
print('-' * 90)
for R in RUNS_ALL:
    label = R['label']
    axes = extract_local_axes(R)
    LOCAL_AXES[label] = axes
    for a in axes:
        print(f'{label:22s}  {a["segment_idx"]:>3d}  {a["peak_ep"]:>6d}  '
              f'{a["L"]:>8.2f}  {int(a["duration"]):>6d}  {a["velocity"]:>10.5f}  '
              f'{a["sigmoidality"]:>+7.3f}  {a["flavor"]:>6s}')
    print()

print()
print('Pairwise angles (degrees, sign-flip invariant) — off-diagonal only:')
for R in RUNS_ALL:
    label = R['label']
    axes = LOCAL_AXES[label]
    if len(axes) < 2:
        print(f'{label:22s}  (n={len(axes)} — no pairs)')
        continue
    M = pairwise_angle_matrix(axes)
    off_diag = M[np.triu_indices_from(M, k=1)]
    print(f'{label:22s}  n={len(axes)}  median={np.median(off_diag):5.1f}°  '
          f'min={off_diag.min():5.1f}°  max={off_diag.max():5.1f}°  '
          f'values={[f"{v:.1f}" for v in off_diag]}')


In [ ]:
# §17.3 — variant-level summary and transit-regime classification.
def classify_regime(mean_dlta, mean_vel, n_segs, med_angle):
    if mean_dlta < 0.010 and mean_vel >= 0.005:
        return 'chord-linear (straight-fast)'
    if mean_dlta < 0.015 and mean_vel < 0.003:
        return 'stuck (low-curvature drift)'
    if mean_dlta >= 0.050 and n_segs >= 2 and (med_angle is not None and med_angle >= 30):
        return 'serial-transit (multi-axis)'
    if 0.020 <= mean_dlta < 0.050:
        return 'single-transit'
    if mean_dlta >= 0.050:
        return 'strong-transit (possibly single-axis)'
    return 'ambiguous'


VARIANT_SUMMARY = {}
print(f'{"variant":22s}  {"class":22s}  {"n":>2s}  {"mean Δ":>7s}  {"mean vel":>9s}  '
      f'{"med ang":>8s}  {"max ang":>8s}  regime')
print('-' * 125)
for R in RUNS_ALL:
    label = R['label']
    axes = LOCAL_AXES[label]
    n = len(axes)
    if n == 0:
        continue
    mean_dlta = float(np.mean([a['sigmoidality'] for a in axes]))
    mean_vel = float(np.mean([a['velocity'] for a in axes]))
    if n >= 2:
        M = pairwise_angle_matrix(axes)
        off = M[np.triu_indices_from(M, k=1)]
        med_angle = float(np.median(off))
        max_angle = float(off.max())
    else:
        med_angle = None
        max_angle = None
    regime = classify_regime(mean_dlta, mean_vel, n, med_angle)
    VARIANT_SUMMARY[label] = dict(
        n_segs=n, mean_sigmoidality=mean_dlta, mean_velocity=mean_vel,
        median_angle=med_angle, max_angle=max_angle, regime=regime,
    )
    ma = f'{med_angle:>7.1f}°' if med_angle is not None else '     —'
    xa = f'{max_angle:>7.1f}°' if max_angle is not None else '     —'
    print(f'{label:22s}  {variant_class_ext[label]:22s}  {n:>2d}  {mean_dlta:>+7.3f}  '
          f'{mean_vel:>9.5f}  {ma:>8s}  {xa:>8s}  {regime}')

print()
print('Regime rules:')
print('  chord-linear         — mean Δ < 0.010 AND mean velocity ≥ 0.005 (straight-fast, p109-like)')
print('  stuck                — mean Δ < 0.015 AND mean velocity < 0.003 (low curvature, low motion)')
print('  single-transit       — 0.020 ≤ mean Δ < 0.050')
print('  serial-transit       — mean Δ ≥ 0.050 AND n_segs ≥ 2 AND median pairwise angle ≥ 30°')
print('  strong-transit       — mean Δ ≥ 0.050 (axes may be near-parallel; not serial)')


In [ ]:
# §17.3 — Δ vs velocity phase scatter. Segregates chord-linear from stuck;
# high-Δ cases lift above the low-Δ band regardless of velocity.
fig_s17c = go.Figure()
for R in RUNS_ALL:
    label = R['label']
    axes = LOCAL_AXES[label]
    if not axes:
        continue
    col = colors_all.get(label, '#333')
    fig_s17c.add_trace(go.Scatter(
        x=[a['velocity'] for a in axes],
        y=[a['sigmoidality'] for a in axes],
        mode='markers+text',
        marker=dict(size=13, color=col, line=dict(width=1, color='white'),
                    symbol=['circle' if a['flavor'] == 'on' else
                            ('square' if a['flavor'] == 'off' else 'diamond') for a in axes]),
        text=[f'seg{a["segment_idx"]}@{a["peak_ep"]}' for a in axes],
        textposition='top center', textfont=dict(size=9, color='#444'),
        name=f'{label} ({variant_class_ext[label]})',
        hovertemplate=(f'{label}<br>seg %{{text}}<br>velocity %{{x:.5f}}'
                       f'<br>Δ %{{y:+.3f}}<extra></extra>'),
    ))

fig_s17c.add_hline(y=0.050, line=dict(color='grey', width=1, dash='dot'),
                   annotation_text='Δ=0.050  serial-transit band', annotation_position='top right')
fig_s17c.add_hline(y=0.020, line=dict(color='grey', width=1, dash='dot'),
                   annotation_text='Δ=0.020  single-transit band', annotation_position='top right')
fig_s17c.add_vline(x=0.005, line=dict(color='grey', width=1, dash='dot'),
                   annotation_text='vel=0.005  straight-fast cutoff', annotation_position='top')
fig_s17c.add_vline(x=0.003, line=dict(color='grey', width=1, dash='dot'),
                   annotation_text='vel=0.003  stuck cutoff', annotation_position='bottom')

fig_s17c.update_layout(
    title=('Transit-axis phase scatter — Δ (sigmoidality) vs velocity (L_local / Δepoch).  '
           'markers: ○ on-arc / □ off-arc / ◇ mix'),
    xaxis=dict(title='velocity  ‖Δparam‖ / Δepoch', type='log'),
    yaxis=dict(title='Δ = R²_sig − R²_lin'),
    height=560, width=900,
)
fig_s17c.show()


In [ ]:
# §17.3 — per-variant pairwise-angle heatmap. Serial-saddle picture predicts
# well-separated off-diagonal values (≥30°). Near-parallel axes → global transit
# was effectively one direction crossed multiple times.
n_variants = len(RUNS_ALL)
fig_s17d = make_subplots(
    rows=n_variants, cols=1,
    subplot_titles=[f'{R["label"]}  ({variant_class_ext[R["label"]]})' for R in RUNS_ALL],
    horizontal_spacing=0.06,
)

cbar_shown = False
for i, R in enumerate(RUNS_ALL):
    label = R['label']
    axes = LOCAL_AXES[label]
    n = len(axes)
    if n == 0:
        continue
    M = pairwise_angle_matrix(axes)
    labels = [f'seg{a["segment_idx"]}@{a["peak_ep"]}' for a in axes]

    fig_s17d.add_trace(go.Heatmap(
        z=M, x=labels, y=labels,
        colorscale='Viridis', zmin=0, zmax=90,
        colorbar=dict(title='angle°', len=0.8) if not cbar_shown else None,
        showscale=not cbar_shown,
        text=[[f'{v:.1f}°' for v in row] for row in M],
        texttemplate='%{text}', textfont=dict(size=10, color='white'),
        hovertemplate='%{x} ↔ %{y}<br>angle %{z:.1f}°<extra></extra>',
    ), row=i+1, col=1)
    cbar_shown = True

    fig_s17d.update_xaxes(tickangle=-35, row=i+1, col=1)

fig_s17d.update_layout(
    height=340 * n_variants, width=450,
    title_text='Pairwise angles between local transit axes (|cos| → sign-flip invariant)',
)
fig_s17d.show()

# Tabular readout — angles, |cos|, and deviation from high-D null.
# In d_mlp ≈ O(10^5) space, two random unit vectors have expected |cos| ≈ 1/√d
# ≈ 0.003, i.e. expected angle ≈ 90°. The interesting quantity is how far below
# 90° the measured pairs sit — i.e. how much direction is shared across segments.
# |cos| is the fraction of direction shared; 90°−angle is the deviation from the
# null (larger deviation = more coupled; ~0° deviation = uncorrelated).
print()
print(f'{"variant":22s}  {"pair":>30s}  {"angle":>7s}  {"|cos|":>6s}  {"90°−ang":>8s}')
print('-' * 85)
for R in RUNS_ALL:
    label = R['label']
    axes = LOCAL_AXES[label]
    if len(axes) < 2:
        continue
    M = pairwise_angle_matrix(axes)
    for i in range(len(axes)):
        for j in range(i + 1, len(axes)):
            ang = M[i, j]
            abs_cos = abs(float(np.cos(np.radians(ang))))
            dev = 90.0 - ang
            pair = (f'seg{axes[i]["segment_idx"]}@{axes[i]["peak_ep"]} ↔ '
                    f'seg{axes[j]["segment_idx"]}@{axes[j]["peak_ep"]}')
            print(f'{label:22s}  {pair:>30s}  {ang:>6.1f}°  {abs_cos:>6.3f}  {dev:>7.1f}°')
    print()


### 17.3 Expected readout and decision rules

**Phase scatter (Δ vs velocity).** The 2-D phase separates the previously-ambiguous low-Δ variants:
- **p109** low-Δ + high-velocity → chord-linear (transit axis is straight, sigmoid degenerates to line).
- **p59** low-Δ + low-velocity → stuck (low curvature because trajectory barely moves, not because it moves straight).
- **p113 / p101r** mid-Δ → single-transit regime; sigmoidal enough to be distinguishable from drift.
- **p101/s999/ds598** high-Δ → serial- or strong-transit depending on angle outcome.

**Pairwise-angle test for serial saddles.** Sharpest read-out is on p101/s999/ds598 (n=3):
- All three off-diagonal angles ≥ 30° → three distinct transit axes → serial saddles confirmed, heteroclinic-chain picture holds for this variant.
- One or more off-diagonal angles ≤ 10° → segments retrace the same direction; the three bumps are one global transit crossed in stages, not three saddles.

**Off-arc-AND-sigmoidal check.** §17.2 surfaced two candidate "first-descent-residue" peaks that are actually sigmoidal:
- p113 canon seg 0 (peak 2000, align 0.14, Δ 0.037)
- p101/s999 seg 0 (peak 2000, align 0.11, Δ 0.057)

If either of these shows a large angle (>30°) to the same variant's on-arc axes, it is a **real saddle that the global arc decomposition was hiding**, not residue. That would reopen §14's assumption that off-arc peaks are first-descent artifacts.

**Intervention design handoff.** Each row of the per-variant axis table gives a concrete `v̂` usable as a parameter-space perturbation direction for the causal test flagged in open question #2 (2026-04-18). Pick the segment with highest Δ and moderate velocity; perturb W at `ep_lo` along `±v̂`; check whether trajectory recovers via the same axis (axis is not load-bearing) or derails (axis is load-bearing).
